# VAZHI 4GB Device Optimization — 270M Gate + QAT + imatrix + Vocabulary Trimming

**Problem:** Gemma 3 1B-it SFT v7.1 (deployment candidate, 96% Tamil word) crashes on 4GB Android
devices at ALL quantization levels (Q4_K_M=762 MiB, Q3_K_M≈693 MiB, Q2_K=652 MiB).

**Root cause:** 262K vocabulary creates ~302M embedding params (30% of model) stored as f32/f16
tensors that don't shrink with quantization → ~600-620 MiB floor regardless of quant method.
Additionally, Flutter/Dart/isolate overhead consumes ~640-950 MB, leaving only ~250-560 MB for
model + compute buffers on 4GB devices.

**Four experiments (fastest-first order):**
1. **Test 0: Gemma 3 270M-it gate** — download bartowski's ~250MB GGUF, test Tamil quality in 15 min. If Tamil is usable, this *is* the 4GB solution (no training needed)
2. **Test 1: Gemma 3 1B QAT gate** — Google's Quantization-Aware Training models (bartowski GGUF). QAT Q2_K at 690MB is *smaller* than v7.1 Q4_K_M (806MB) with potentially better quality because quantization noise was baked into training itself. This is the "Quantization-Aware Training" that Lessons Learned Phase 5 identified as a missed approach — Google did it for us
3. **Part A: imatrix quantization** — improve Tamil quality at aggressive quant levels + test embed/output tensor quant flags (for 6GB+ quality improvement, and to test if the "non-shrinking floor" can move)
4. **Part B: Vocabulary trimming** — prune 262K→~50K vocab to cut embedding floor by ~466 MiB (the proven path to 4GB if both 270M and QAT fail)

**Decision logic:**
- If 270M Tamil is acceptable → ship 270M as 4GB tier, skip Part B
- If QAT Q2_K (690MB) produces >90% Tamil word → potential 4GB candidate AND better 6GB+ tier
- If QAT works but OOMs on 4GB → still upgrade 6GB+ tier from v7.1 to QAT
- Part A imatrix always useful (improves 6GB+ tier quality on v7.1 regardless)
- Part B vocab trimming is the fallback if all else fails for 4GB

**Two-tier deployment strategy (all analyses agree):**
- **4GB devices:** Hybrid-only by default. Optional LLM mode gated by preflight memory check
- **6GB+ devices:** Best of {v7.1 imatrix Q4_K_M, QAT Q4_K_M/Q3_K_M} — whichever has best Tamil

**Model:** `CryptoYogi/vazhi-v7_1` (Gemma 3 1B-it + SFT v7.0 r=8 + SFT v7.1 r=16)

**Runtime:** Colab Pro GPU (L4/A100 recommended). ~5 hours total.

In [1]:
# Cell 1 — Dependencies + GPU Check
!pip install -q -U "transformers>=4.45.0,<5.0.0" huggingface_hub accelerate sentencepiece protobuf

import torch, sys
print(f"✅ Python: {sys.version.split()[0]}")
print(f"✅ PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', 0)
    print(f"   VRAM: {vram / 1024**3:.0f} GB")
else:
    print("⚠️  No GPU — imatrix generation and inference will use CPU (slower)")
print()
!free -h | head -2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 31.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.5 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
✅ Python: 3.12.12
✅ PyTorch: 2.9.0+cu128
   CUDA: True


In [2]:
# Cell 2 — HuggingFace Login
from huggingface_hub import notebook_login
notebook_login()
print("✅ Logged in to HuggingFace")

✅ Logged in to HuggingFace


In [3]:
# Cell 3 — Configuration + Tamil Quality Functions
#
# Reused from Vazhi_Model_Comparison_v1.ipynb (Cell 4-5) and
# Vazhi_GGUF_v7_1_Gemma3.ipynb (Cell 4).

import os, gc, re, json, subprocess
import numpy as np

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# === MODEL ===
HF_MODEL = "CryptoYogi/vazhi-v7_1"       # Merged fp16/bf16 model on HuggingFace
BASE_MODEL = "google/gemma-3-1b-it"       # Original base (for tokenizer reference)
LOCAL_MODEL_DIR = "./vazhi-v7_1"           # Local download directory
F16_GGUF = "vazhi-v7.1-f16.gguf"          # Intermediate f16 GGUF

# === GENERATION ===
MAX_NEW_TOKENS = 200
TEMPERATURE = 0.7

# Gemma 3 prompt format — no system role, embed identity in user turn
SYSTEM_CONTEXT = "நீங்கள் வழி (VAZHI), தமிழ்நாட்டு மக்களுக்கான AI உதவியாளர். நீங்கள் தமிழில் பதிலளிப்பீர்கள்."

# === TAMIL QUALITY FUNCTIONS ===

def tamil_char_pct(text):
    """% of non-whitespace, non-digit chars that are Tamil Unicode."""
    if not text:
        return 0.0
    total = sum(1 for c in text if not c.isspace() and not c.isdigit())
    if total == 0:
        return 0.0
    tamil = sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')
    return 100.0 * tamil / total

def tamil_word_score(text):
    """Score based on per-word Tamil character majority. Returns (pct, count, total)."""
    words = text.split()
    if not words:
        return 0.0, 0, 0
    tamil_words = 0
    for w in words:
        clean = re.sub(r'[\d\W]', '', w)
        if not clean:
            continue
        tamil_chars = sum(1 for c in clean if '\u0B80' <= c <= '\u0BFF')
        if tamil_chars / len(clean) > 0.5:
            tamil_words += 1
    return 100.0 * tamil_words / max(len(words), 1), tamil_words, len(words)

def compute_repeat_ratio(text, n=3):
    """Detect repetitive output via trigram ratio. 0=unique, 1=fully repetitive."""
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    if not ngrams:
        return 0.0
    return 1.0 - len(set(ngrams)) / len(ngrams)

# === EVAL PROMPTS (from Model Comparison v1) ===
TAMIL_PROMPTS = [
    {"text": "வணக்கம்",                                          "cat": "greeting",  "desc": "Basic greeting"},
    {"text": "நீங்கள் யார்?",                                    "cat": "identity",  "desc": "Who are you?"},
    {"text": "காலையில் என்ன சாப்பிடலாம்?",                        "cat": "health",    "desc": "Morning food advice"},
    {"text": "ரேஷன் கார்டு பற்றி தகவல் தேவை",                    "cat": "govt",      "desc": "Ration card info"},
    {"text": "திருக்குறள் பற்றி சொல்லுங்கள்",                    "cat": "culture",   "desc": "About Thirukkural"},
    {"text": "ஒரு தெரியாத எண்ணில் இருந்து மெசேஜ் வந்தது",       "cat": "safety",    "desc": "Unknown message scam"},
    {"text": "முதியோர் ஓய்வூதியம் பற்றி சொல்லுங்கள்",           "cat": "govt",      "desc": "Old age pension"},
    {"text": "நீரிழிவு நோய் பற்றி சொல்லுங்கள்",                 "cat": "health",    "desc": "About diabetes"},
    {"text": "கல்வி கடன் பற்றி தகவல்",                           "cat": "education", "desc": "Education loan info"},
    {"text": "சைபர் மோசடியில் இருந்து பணம் இழந்தால் என்ன செய்யலாம்?", "cat": "security", "desc": "Cyber fraud help"},
]

ENGLISH_PROMPTS = [
    {"text": "What is your name?",            "cat": "english", "desc": "English identity"},
    {"text": "Tell me about Tamil Nadu",       "cat": "english", "desc": "English knowledge"},
    {"text": "How do I apply for a passport?", "cat": "english", "desc": "English practical"},
]

ALL_PROMPTS = TAMIL_PROMPTS + ENGLISH_PROMPTS

# === GGUF INFERENCE HELPER ===

def build_gemma_prompt(user_msg):
    """Build Gemma 3 prompt with VAZHI identity embedded in user turn."""
    return (
        f"<start_of_turn>user\n"
        f"{SYSTEM_CONTEXT}\n\n"
        f"{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )

def run_gguf_inference(gguf_path, prompt_text, max_tokens=200, use_gpu=True):
    """Run llama-cli inference on a GGUF file and return generated text."""
    ngl = "99" if use_gpu and torch.cuda.is_available() else "0"
    cmd = [
        "./llama.cpp/build/bin/llama-cli",
        "-m", gguf_path,
        "-p", prompt_text,
        "-n", str(max_tokens),
        "--temp", str(TEMPERATURE),
        "-ngl", ngl,
        "--no-display-prompt",
        "--log-disable",
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
        output = result.stdout.strip()
        # Remove trailing special tokens
        for tok in ["<end_of_turn>", "<eos>", "<start_of_turn>"]:
            output = output.split(tok)[0]
        return output.strip()
    except subprocess.TimeoutExpired:
        return "[TIMEOUT]"
    except Exception as e:
        return f"[ERROR: {e}]"

def eval_gguf(gguf_path, label, prompts=None, use_gpu=True):
    """Evaluate a GGUF model on all prompts, return results dict."""
    if prompts is None:
        prompts = ALL_PROMPTS

    print(f"\n{'='*65}")
    print(f"  📊 EVALUATING: {label}")
    print(f"  GGUF: {gguf_path}")
    print(f"{'='*65}")

    size_mb = os.path.getsize(gguf_path) / 1e6
    print(f"  Size: {size_mb:.1f} MB")

    results = []
    for item in prompts:
        prompt = build_gemma_prompt(item['text'])
        resp = run_gguf_inference(gguf_path, prompt, max_tokens=MAX_NEW_TOKENS, use_gpu=use_gpu)

        t_char = tamil_char_pct(resp)
        t_word, _, _ = tamil_word_score(resp)
        rep = compute_repeat_ratio(resp)

        results.append({
            'prompt': item['text'],
            'category': item['cat'],
            'description': item['desc'],
            'response': resp,
            'tamil_char_pct': t_char,
            'tamil_word_pct': t_word,
            'repeat_ratio': rep,
        })

        print(f"\n  [{item['cat']:>10}] Char:{t_char:.0f}% Word:{t_word:.0f}% Rep:{rep:.2f}")
        print(f"    Q: {item['text']}")
        print(f"    A: {resp[:300]}")

    # Summary (Tamil prompts only)
    tamil_results = [r for r in results if r['category'] != 'english']
    avg_char = np.mean([r['tamil_char_pct'] for r in tamil_results]) if tamil_results else 0
    avg_word = np.mean([r['tamil_word_pct'] for r in tamil_results]) if tamil_results else 0
    avg_rep = np.mean([r['repeat_ratio'] for r in tamil_results]) if tamil_results else 0
    non_empty = sum(1 for r in results if len(r['response'].strip()) >= 10)

    print(f"\n  {'─'*50}")
    print(f"  📊 {label} SUMMARY (Tamil prompts only):")
    print(f"     Avg Tamil char: {avg_char:.1f}%")
    print(f"     Avg Tamil word: {avg_word:.1f}%")
    print(f"     Avg repeat:     {avg_rep:.2f}")
    print(f"     Non-empty:      {non_empty}/{len(results)}")
    print(f"     Size:           {size_mb:.1f} MB")

    return {
        'label': label,
        'gguf_path': gguf_path,
        'size_mb': size_mb,
        'results': results,
        'avg_char': avg_char,
        'avg_word': avg_word,
        'avg_rep': avg_rep,
        'non_empty': non_empty,
    }

print(f"✅ Config ready")
print(f"   Model: {HF_MODEL}")
print(f"   Eval prompts: {len(ALL_PROMPTS)} ({len(TAMIL_PROMPTS)} Tamil + {len(ENGLISH_PROMPTS)} English)")

✅ Config ready
   Model: CryptoYogi/vazhi-v7_1
   Eval prompts: 13 (10 Tamil + 3 English)


In [4]:
# Cell 4 — Build llama.cpp + Download Model + Convert to f16 GGUF
#
# llama.cpp is needed for: imatrix generation, quantization, and inference.
# cmake build required for Gemma 3 architecture support.

import os

# --- Build llama.cpp ---
if not os.path.exists("llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
else:
    print("llama.cpp already cloned")

# Build with cmake + CUDA (for GPU-accelerated imatrix generation)
has_cuda = torch.cuda.is_available()
if has_cuda:
    !cd llama.cpp && cmake -B build -DGGML_CUDA=ON 2>&1 | tail -3
    !cd llama.cpp && cmake --build build --config Release -j$(nproc) 2>&1 | tail -5
else:
    !cd llama.cpp && cmake -B build 2>&1 | tail -3
    !cd llama.cpp && cmake --build build --config Release -j$(nproc) 2>&1 | tail -5

# Install Python requirements for convert_hf_to_gguf.py
!pip install -q -r llama.cpp/requirements.txt 2>&1 | tail -2

# Verify key binaries exist
for binary in ["llama-cli", "llama-quantize", "llama-imatrix"]:
    path = f"llama.cpp/build/bin/{binary}"
    if os.path.exists(path):
        print(f"✅ {binary}")
    else:
        print(f"❌ {binary} NOT FOUND — build failed?")

# --- Download v7.1 merged model from HuggingFace ---
print(f"\n--- Downloading {HF_MODEL} ---")
from huggingface_hub import snapshot_download

if not os.path.exists(LOCAL_MODEL_DIR):
    snapshot_download(
        repo_id=HF_MODEL,
        local_dir=LOCAL_MODEL_DIR,
        ignore_patterns=["*.md", "*.txt"],
    )
    print(f"✅ Model downloaded to {LOCAL_MODEL_DIR}")
else:
    print(f"✅ Model already exists at {LOCAL_MODEL_DIR}")

!ls -lh {LOCAL_MODEL_DIR}/*.safetensors {LOCAL_MODEL_DIR}/config.json 2>/dev/null | head -10

# --- Convert to f16 GGUF ---
print(f"\n--- Converting to f16 GGUF ---")
if not os.path.exists(F16_GGUF):
    !python llama.cpp/convert_hf_to_gguf.py {LOCAL_MODEL_DIR} --outfile {F16_GGUF} --outtype f16
    print(f"✅ Converted: {F16_GGUF}")
else:
    print(f"✅ f16 GGUF already exists: {F16_GGUF}")

size_bytes = os.path.getsize(F16_GGUF)
print(f"   Size: {size_bytes / 1e6:.1f} MB ({size_bytes / 1e9:.2f} GB)")

Cloning into 'llama.cpp'...
remote: Enumerating objects: 2545, done.
remote: Counting objects: 100% (2545/2545), done.
remote: Compressing objects: 100% (2028/2028), done.
remote: Total 2545 (delta 511), reused 1669 (delta 444), pack-reused 0 (from 0)
Receiving objects: 100% (2545/2545), 27.52 MiB | 19.14 MiB/s, done.
Resolving deltas: 100% (511/511), done.
-- Configuring done (11.3s)
-- Generating done (0.3s)
-- Build files have been written to: /content/llama.cpp/build
^C
❌ llama-cli NOT FOUND — build failed?
❌ llama-quantize NOT FOUND — build failed?
❌ llama-imatrix NOT FOUND — build failed?

--- Downloading CryptoYogi/vazhi-v7_1 ---


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

✅ Model downloaded to ./vazhi-v7_1
-rw-r--r-- 1 root root 1.6K Feb 17 23:35 ./vazhi-v7_1/config.json
-rw-r--r-- 1 root root 1.9G Feb 17 23:36 ./vazhi-v7_1/model.safetensors

--- Converting to f16 GGUF ---
INFO:hf-to-gguf:Loading model: vazhi-v7_1
INFO:hf-to-gguf:Model architecture: Gemma3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.float16 --> F16, shape = {1152, 262144}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.float16 --> F32, shape = {1152}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.float16 --> F16, shape = {6912, 1152}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.float16 --> F16, shape = {1152, 6912}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.float16 --> F16, shape = {1152, 6912}
INFO:hf-to-gguf:blk.0.post_attention_norm.weight,  torch.float

In [5]:
  # Fix: Build only the llama.cpp tools we need
  %cd /content/llama.cpp
  !cmake --build build --target llama-cli llama-quantize llama-imatrix --config Release -j$(nproc)
  %cd /content

  # Verify
  import os
  for tool in ["llama-cli", "llama-quantize", "llama-imatrix"]:
      path = f"/content/llama.cpp/build/bin/{tool}"
      if os.path.exists(path):
          print(f"✅ {tool}")
      else:
          print(f"❌ {tool} — NOT FOUND")


/content/llama.cpp
[  0%] Built target build_info
[  0%] Built target cpp-httplib
[  3%] Built target ggml-base
[  6%] Built target ggml-cpu
[  6%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq4_xs.cu.o
[  6%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq4_nl.cu.o
[  6%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-mxfp4.cu.o
[  6%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-q2_k.cu.o
[  7%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-q3_k.cu.o
[  7%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-q4_0.cu.o
[  7%] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-q4_1.cu.o
[  7%] Building CUDA object ggml/src/ggml-cuda/CMakeFile

In [6]:
  # Stop the current build first (Runtime > Interrupt), then run this:
  %cd /content/llama.cpp
  !cmake -B build -DGGML_CUDA=OFF    # Reconfigure without CUDA
  !cmake --build build --target llama-cli llama-quantize llama-imatrix --config Release -j$(nproc)
  %cd /content

  # Verify
  import os
  for tool in ["llama-cli", "llama-quantize", "llama-imatrix"]:
      path = f"/content/llama.cpp/build/bin/{tool}"
      print(f"{'✅' if os.path.exists(path) else '❌'} {tool}")


/content/llama.cpp
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.9.7
-- ggml commit:  e2f19b3
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: common
-- Configuring done (0.2s)
-- Generating done (0.4s)
-- Build files have been written to: /content/llama.cpp/build
[  0%] Built target build_info
[  3%] Built target ggml-base
[  3%] Built target cpp-httplib
[ 11%] Built target ggml-cpu
[ 11%] Building CXX object ggml/src/CMakeFiles/ggml.dir/ggml-backend-dl.cpp.o
[ 13%] Building CXX object ggml/src/CMakeFiles/ggml.dir/ggml-backend-reg.cpp.o
[ 13%] Linking CXX shared library ../../bin/libggml.so
[ 13%] Built target ggml
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama.cpp.o
[ 13%] Buildin

---

# Test 0: Gemma 3 270M-it — Quick Tamil Quality Gate

**Goal:** Determine if Gemma 3 270M-it can produce acceptable Tamil at ~250 MB GGUF size.
If yes, this *is* the 4GB solution — no vocab trimming needed.

**Why this model:**
- 270M params with 262K vocab, hidden_size=640, 18 layers
- ~168M embedding params + ~100M transformer params
- Q4_K_M GGUF ≈ 253 MB (bartowski) — well within 4GB device budget (~250-560 MB available)
- Same 262K multilingual vocab as Gemma 3 1B → native Tamil token coverage
- bartowski's variants already use imatrix quantization with calibration data
- `_L` variants (Q6_K_L, Q4_K_L) use Q8_0 for embed/output weights — the best quality baseline

**Variants tested:**
- Q4_K_M (253 MB) — main 4GB candidate
- IQ4_NL (242 MB) — ARM-optimized, slightly smaller
- Q6_K_L (280 MB) — Q8_0 embed/output, quality ceiling test

**Risk:** Small transformer capacity (~100M non-embedding params) may produce shallow/repetitive
Tamil. But for "language glue" (routing, paraphrasing, short explanations) alongside hybrid
knowledge packs, it could be sufficient.

**Time:** ~15 minutes (download + inference, no training)

In [8]:
  # Install llama-cpp-python with pre-built CUDA support (~2 min)
  !pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

  # Patch inference to use Python GPU API instead of CLI
  from llama_cpp import Llama

  _loaded_models = {}

  def run_gguf_inference(gguf_path, prompt_text, max_tokens=200, use_gpu=True):
      """GPU-accelerated inference via llama-cpp-python."""
      try:
          if gguf_path not in _loaded_models:
              _loaded_models[gguf_path] = Llama(
                  model_path=gguf_path,
                  n_gpu_layers=99 if use_gpu else 0,
                  n_ctx=2048,
                  verbose=False,
              )
          llm = _loaded_models[gguf_path]
          result = llm(prompt_text, max_tokens=max_tokens, temperature=0.7, stop=["<end_of_turn>", "<eos>"])
          return result["choices"][0]["text"].strip()
      except Exception as e:
          return f"[ERROR: {e}]"

  print("✅ GPU inference ready via llama-cpp-python")


Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.3/551.3 MB 844.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
✅ GPU inference ready via llama-cpp-python


In [9]:
# Test 0 — Gemma 3 270M-it Tamil Quality Gate
#
# Download bartowski's pre-quantized GGUF and run our standard eval.
# No training, no quantizing — pure "does this model speak Tamil?" test.
#
# All bartowski 270M variants already use imatrix quantization.
# The _L variants use Q8_0 for embed/output weights.
#
# Reference: https://huggingface.co/bartowski/google_gemma-3-270m-it-GGUF

from huggingface_hub import hf_hub_download
import os

# --- Download 270M GGUFs ---
GGUF_270M_REPO = "bartowski/google_gemma-3-270m-it-GGUF"

# Three variants: main candidate, ARM-optimized, and quality ceiling
variants_270m = {
    "270M Q4_K_M": "google_gemma-3-270m-it-Q4_K_M.gguf",      # 253 MB, main candidate
    "270M IQ4_NL": "google_gemma-3-270m-it-IQ4_NL.gguf",      # 242 MB, ARM-optimized
    "270M Q6_K_L": "google_gemma-3-270m-it-Q6_K_L.gguf",      # 280 MB, Q8_0 embed/output (quality ceiling)
}

print("Downloading Gemma 3 270M-it GGUFs...")
local_270m = {}
for label, filename in variants_270m.items():
    local_path = filename
    if not os.path.exists(local_path):
        try:
            hf_hub_download(
                repo_id=GGUF_270M_REPO,
                filename=filename,
                local_dir=".",
            )
            print(f"  ✅ {label}: {filename} ({os.path.getsize(local_path)/1e6:.0f} MB)")
        except Exception as e:
            print(f"  ⚠️ {label} download failed: {e}")
            print(f"     Try manually: huggingface-cli download {GGUF_270M_REPO} {filename}")
            continue
    else:
        print(f"  ✅ {label}: already exists ({os.path.getsize(local_path)/1e6:.0f} MB)")
    local_270m[label] = local_path

# --- Run Tamil quality gate ---
print(f"\n{'='*70}")
print(f"  GEMMA 3 270M-it — TAMIL QUALITY GATE")
print(f"{'='*70}")

results_270m = {}
for label, path in local_270m.items():
    result = eval_gguf(path, label)
    results_270m[label] = result

# --- GO/NO-GO verdict ---
print(f"\n\n{'='*70}")
print(f"  TEST 0 VERDICT: Gemma 3 270M-it")
print(f"{'='*70}")

for label, r in results_270m.items():
    print(f"\n  {label}:")
    print(f"    Size:        {r['size_mb']:.0f} MB")
    print(f"    Tamil char:  {r['avg_char']:.1f}%")
    print(f"    Tamil word:  {r['avg_word']:.1f}%")
    print(f"    Repeat:      {r['avg_rep']:.2f}")
    print(f"    Non-empty:   {r['non_empty']}/{len(r['results'])}")

# Check Q6_K_L as quality ceiling
q6kl = results_270m.get("270M Q6_K_L")
if q6kl:
    print(f"\n  Q6_K_L is the quality ceiling (Q8_0 embed/output, 280 MB):")
    print(f"    If Q6_K_L can't produce Tamil, 270M is fundamentally limited")
    print(f"    If Q6_K_L is good but Q4_K_M isn't, embed/output precision matters")

# Decision
best_270m = max(results_270m.values(), key=lambda x: x['avg_word']) if results_270m else None

if best_270m and best_270m['avg_word'] >= 70:
    print(f"\n  ✅ GO — 270M produces usable Tamil ({best_270m['avg_word']:.0f}% word)")
    print(f"     → This could be the 4GB tier model")
    print(f"     → Consider light SFT with v7.x pipeline for VAZHI personality")
    print(f"     → Still run Test 1 (QAT) and Part A (imatrix for 6GB+ tier)")
    print(f"     → Part B (vocab trimming) may be unnecessary — evaluate after human review")
    SKIP_VOCAB_TRIMMING = False  # Still run it, but it's lower priority
elif best_270m and best_270m['avg_word'] >= 40:
    print(f"\n  ⚠️ MARGINAL — 270M has some Tamil ({best_270m['avg_word']:.0f}% word)")
    print(f"     → May work with SFT fine-tuning")
    print(f"     → Proceed with Test 1 (QAT), Part A, and Part B as planned")
    SKIP_VOCAB_TRIMMING = False
else:
    word_pct = best_270m['avg_word'] if best_270m else 0
    print(f"\n  ❌ NO-GO — 270M Tamil is insufficient ({word_pct:.0f}% word)")
    print(f"     → 270M eliminated for 4GB tier")
    print(f"     → Proceed with Test 1 (QAT), Part A and Part B")
    SKIP_VOCAB_TRIMMING = False

print(f"\n  For comparison:")
print(f"    v7.1 original (Gemma 3 1B): ~96% Tamil word (from training eval)")
print(f"    Human review of 270M outputs above is critical — metrics can be misleading")

  ✅ 270M Q4_K_M: already exists (253 MB)
  ✅ 270M IQ4_NL: already exists (242 MB)
  ✅ 270M Q6_K_L: already exists (283 MB)

  GEMMA 3 270M-it — TAMIL QUALITY GATE

  📊 EVALUATING: 270M Q4_K_M
  GGUF: google_gemma-3-270m-it-Q4_K_M.gguf
  Size: 253.1 MB


llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



  [  greeting] Char:94% Word:100% Rep:0.00
    Q: வணக்கம்
    A: வணக்கம்! நீங்கள் எப்படி இருக்கிறீர்கள்?

  [  identity] Char:89% Word:100% Rep:0.00
    Q: நீங்கள் யார்?
    A: நான் யார்?

  [    health] Char:95% Word:92% Rep:0.00
    Q: காலையில் என்ன சாப்பிடலாம்?
    A: வணக்கம்! நான் உங்களுக்கு AI உதவியாளர். உங்கள் விருப்பத்திற்கு ஏற்ப நீங்கள் என்ன சாப்பிடலாம் என்பதைத் தீர்மானிக்கலாம்.

  [      govt] Char:69% Word:71% Rep:0.00
    Q: ரேஷன் கார்டு பற்றி தகவல் தேவை
    A: ரேஷன் கார்டு (Visa Card) பற்றி தகவல் தேவை.

  [   culture] Char:96% Word:100% Rep:0.00
    Q: திருக்குறள் பற்றி சொல்லுங்கள்
    A: திருக்குறள் பற்றி சொல்லுங்கள்.

  [    safety] Char:95% Word:100% Rep:0.00
    Q: ஒரு தெரியாத எண்ணில் இருந்து மெசேஜ் வந்தது
    A: வணக்கம்! உங்கள் கேள்விக்கு பதில் அளிக்கிறேன்.

  [      govt] Char:90% Word:94% Rep:0.26
    Q: முதியோர் ஓய்வூதியம் பற்றி சொல்லுங்கள்
    A: உங்களிடம் முதியோர் ஓய்வூதியம் பற்றி சொல்லும் பதில் இதோ:

முதியோர் ஓய்வூதியம் என்பது, ஒரு காலத்திற்குள், ஒரு பகுதியில் (

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



  [  greeting] Char:88% Word:100% Rep:0.00
    Q: வணக்கம்
    A: வணக்கம்!

  [  identity] Char:83% Word:75% Rep:0.00
    Q: நீங்கள் யார்?
    A: வணக்கம்! நான் AI உதவியாளர்.

  [    health] Char:88% Word:100% Rep:0.00
    Q: காலையில் என்ன சாப்பிடலாம்?
    A: வணக்கம்!

  [      govt] Char:96% Word:100% Rep:0.00
    Q: ரேஷன் கார்டு பற்றி தகவல் தேவை
    A: ரேஷன் கார்டு பற்றி தகவல் தேவை.

  [   culture] Char:97% Word:97% Rep:0.10
    Q: திருக்குறள் பற்றி சொல்லுங்கள்
    A: சாரி, தமிழ்நாட்டு மக்களுக்கான AI உதவியாளர்.

உங்களுடைய கருத்துக்களைப் புரிந்து கொள்ள நான் உங்களுக்கு உதவ விரும்புகிறேன்.

உங்களுடைய கருத்துக்களைப் புரிந்து கொண்டு, அவற்றைச் சிறப்பாகச் செய்ய நான் உங்களுக்கு உதவ விரும்புகிறேன்.

உங்களுக்கான உதவிக்காக, நீங்கள் என்ன பற்றி பேச விரும்புகிறீர்கள்?

  [    safety] Char:88% Word:100% Rep:0.00
    Q: ஒரு தெரியாத எண்ணில் இருந்து மெசேஜ் வந்தது
    A: வணக்கம்!
வணக்கம்.

  [      govt] Char:94% Word:96% Rep:0.04
    Q: முதியோர் ஓய்வூதியம் பற்றி சொல்லுங்கள்
    A: வணக்கம்!

உங்களுடைய க

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



  [  greeting] Char:94% Word:100% Rep:0.00
    Q: வணக்கம்
    A: வணக்கம்! நீங்கள் எப்படி இருக்கிறீர்கள்?

  [  identity] Char:97% Word:100% Rep:0.00
    Q: நீங்கள் யார்?
    A: நான் ஒரு பெரிய மொழி மாதிரி. நான் கூகிாவில் பயிற்சி அளிக்கப்பட்டது.

  [    health] Char:98% Word:100% Rep:0.00
    Q: காலையில் என்ன சாப்பிடலாம்?
    A: காலையில் உங்களுக்கு பிடித்தமான உணவை பற்றி சொல்லுங்கள்! நான் உங்களுக்கு உதவ தயாராக இருக்கிறேன்.

  [      govt] Char:96% Word:100% Rep:0.00
    Q: ரேஷன் கார்டு பற்றி தகவல் தேவை
    A: ரேஷன் கார்டு பற்றி தகவல் தேவை.

  [   culture] Char:94% Word:95% Rep:0.05
    Q: திருக்குறள் பற்றி சொல்லுங்கள்
    A: திருக்குறள் பற்றி நீங்கள் கேட்கிறீர்கள் என்று நினைக்கிறேன். திருக்குறள் என்பது ஒரு முக்கியமான மற்றும் பல்துறை সাহিত্য முறையாகும். இது தமிழ்நாட்டில் உள்ள பல எழுத்தாளர்கள் மற்றும் ஆராய்ச்சியாளர்கள் உருவாக்கியது. திருக்குறள் ஒரு தத்துவார்த்த மற்றும் ஆன்மீக வடிவம். இது, கடவுளின் சக்தியையும், ஆன்மாக்களின

  [    safety] Char:97% Word:100% Rep:0.00
    Q: ஒரு தெரியாத எண்ணி

---

# Test 1: Gemma 3 1B QAT — Quantization-Aware Training Gate

**Goal:** Test Google's QAT (Quantization-Aware Training) variants of Gemma 3 1B-it. These
models were trained with quantization noise baked in, so they should produce better output at
aggressive quant levels than post-training-quantized models.

**Why QAT changes the calculus:**
- Your entire Phase 4-26 journey was fighting **post-training quantization** destroying Tamil
- Google has now done QAT on Gemma 3 1B-it: the model *learned to be robust* to quant errors
- QAT Q2_K at 690MB is **smaller** than v7.1 Q4_K_M (806MB) — potentially better quality too
- This is exactly the "Quantization-Aware Training" that Lessons Learned Phase 5 identified
  as a missed approach — Google did it for us

**Key insight:** If QAT Q2_K produces >90% Tamil word score at 690MB, it could:
1. Replace v7.1 Q4_K_M as the 6GB+ tier model (better quality, smaller file)
2. Potentially fit 4GB devices (690MB model + ~640-950MB Flutter overhead is tight but possible)

**Variants tested (from bartowski, all imatrix-quantized):**
- Q4_K_M (810 MB) — direct comparison vs v7.1 Q4_K_M (806 MB)
- Q3_K_M (720 MB) — aggressive but QAT-robust
- IQ3_M (700 MB) — imatrix-optimized aggressive quant
- Q2_K (690 MB) — most aggressive, "surprisingly usable" per bartowski

**Important:** These are vanilla Gemma 3 1B-it QAT — NOT your v7.1 SFT. Tamil quality comes
from Gemma's pretrained multilingual capability, not fine-tuning. If QAT Tamil is good, you
could SFT on top of the QAT base for even better results.

**Time:** ~20 minutes (download + inference)

In [10]:
# Test 1 — Gemma 3 1B QAT Tamil Quality Gate
#
# Google's Quantization-Aware Training models: trained with quantization
# noise in the loop, so quality at Q2_K/Q3_K should be much better than
# post-training quantization on the same architecture.
#
# Reference: https://huggingface.co/bartowski/google_gemma-3-1b-it-qat-GGUF
# Blog: https://developers.googleblog.com/en/gemma-3-quantized-aware-trained-state-of-the-art-ai-to-consumer-gpus/

from huggingface_hub import hf_hub_download
import os

# --- Download QAT GGUFs ---
GGUF_QAT_REPO = "bartowski/google_gemma-3-1b-it-qat-GGUF"

variants_qat = {
    "QAT Q4_K_M": "google_gemma-3-1b-it-qat-Q4_K_M.gguf",   # 810 MB — compare vs v7.1 Q4_K_M (806 MB)
    "QAT Q3_K_M": "google_gemma-3-1b-it-qat-Q3_K_M.gguf",   # 720 MB
    "QAT IQ3_M":  "google_gemma-3-1b-it-qat-IQ3_M.gguf",    # 700 MB
    "QAT Q2_K":   "google_gemma-3-1b-it-qat-Q2_K.gguf",     # 690 MB — smallest, key test
}

print("Downloading Gemma 3 1B QAT GGUFs...")
print("(These are Google's official QAT models, quantized by bartowski with imatrix)")
local_qat = {}
for label, filename in variants_qat.items():
    local_path = filename
    if not os.path.exists(local_path):
        try:
            hf_hub_download(
                repo_id=GGUF_QAT_REPO,
                filename=filename,
                local_dir=".",
            )
            print(f"  ✅ {label}: {filename} ({os.path.getsize(local_path)/1e6:.0f} MB)")
        except Exception as e:
            print(f"  ⚠️ {label} download failed: {e}")
            print(f"     Try manually: huggingface-cli download {GGUF_QAT_REPO} {filename}")
            continue
    else:
        print(f"  ✅ {label}: already exists ({os.path.getsize(local_path)/1e6:.0f} MB)")
    local_qat[label] = local_path

# --- Run Tamil quality eval ---
print(f"\n{'='*70}")
print(f"  GEMMA 3 1B QAT — TAMIL QUALITY EVALUATION")
print(f"{'='*70}")

results_qat = {}
for label, path in local_qat.items():
    result = eval_gguf(path, label)
    results_qat[label] = result

# --- Compare QAT vs v7.1 ---
print(f"\n\n{'='*70}")
print(f"  TEST 1 VERDICT: Gemma 3 1B QAT vs v7.1")
print(f"{'='*70}")

print(f"\n  Reference: v7.1 Q4_K_M = 806 MB, ~96% Tamil word (training eval)")
print(f"")

for label, r in sorted(results_qat.items(), key=lambda x: x[1]['size_mb']):
    print(f"  {label}:")
    print(f"    Size:        {r['size_mb']:.0f} MB")
    print(f"    Tamil char:  {r['avg_char']:.1f}%")
    print(f"    Tamil word:  {r['avg_word']:.1f}%")
    print(f"    Repeat:      {r['avg_rep']:.2f}")
    print(f"    Non-empty:   {r['non_empty']}/{len(r['results'])}")
    # Compare to v7.1
    size_diff = r['size_mb'] - 806
    print(f"    vs v7.1:     {'+' if size_diff >= 0 else ''}{size_diff:.0f} MB")
    print()

# Key comparison: QAT Q4_K_M vs v7.1 Q4_K_M (same size, different quality?)
qat_q4 = results_qat.get("QAT Q4_K_M")
if qat_q4:
    print(f"  HEAD-TO-HEAD: QAT Q4_K_M ({qat_q4['size_mb']:.0f}MB) vs v7.1 Q4_K_M (806MB)")
    print(f"    QAT Tamil word:  {qat_q4['avg_word']:.1f}%")
    print(f"    v7.1 Tamil word: ~96% (from training eval)")
    if qat_q4['avg_word'] >= 90:
        print(f"    → QAT maintains Tamil quality — could replace v7.1 for 6GB+ tier")
    elif qat_q4['avg_word'] >= 70:
        print(f"    → QAT has decent Tamil — SFT on QAT base could match/exceed v7.1")
    else:
        print(f"    → QAT Tamil is weaker — v7.1 SFT still needed for quality")

# Key test: QAT Q2_K as potential 4GB candidate
qat_q2 = results_qat.get("QAT Q2_K")
if qat_q2:
    print(f"\n  4GB CANDIDATE: QAT Q2_K ({qat_q2['size_mb']:.0f}MB)")
    print(f"    Tamil word:  {qat_q2['avg_word']:.1f}%")
    if qat_q2['avg_word'] >= 90:
        print(f"    ✅ EXCELLENT — QAT Q2_K has strong Tamil at {qat_q2['size_mb']:.0f}MB")
        print(f"       → 116MB smaller than v7.1 Q4_K_M, potentially better quality")
        print(f"       → Test on 4GB device: {qat_q2['size_mb']:.0f}MB model + ~640-950MB Flutter overhead")
        print(f"       → If it fits, this solves BOTH quality and memory simultaneously")
    elif qat_q2['avg_word'] >= 70:
        print(f"    ⚠️ USABLE — QAT Q2_K has acceptable Tamil at {qat_q2['size_mb']:.0f}MB")
        print(f"       → Worth testing on 4GB device")
        print(f"       → Consider SFT on QAT base for better Tamil")
    else:
        print(f"    ❌ INSUFFICIENT — QAT Q2_K Tamil too low for deployment")
        print(f"       → QAT doesn't fix Tamil at aggressive quant levels")
        print(f"       → Proceed with Part A (imatrix on v7.1) and Part B (vocab trimming)")

# Best QAT candidate
best_qat = max(results_qat.values(), key=lambda x: x['avg_word']) if results_qat else None
if best_qat:
    print(f"\n  BEST QAT: {best_qat['avg_word']:.0f}% Tamil word at {best_qat['size_mb']:.0f} MB")

print(f"\n  NOTE: QAT models are vanilla Gemma 3 1B-it (not your v7.1 SFT).")
print(f"  If QAT quality is good, you could do SFT v7.x on QAT base for even better results.")
print(f"  Human review of QAT outputs is critical — compare semantic quality, not just metrics.")

(These are Google's official QAT models, quantized by bartowski with imatrix)


google_gemma-3-1b-it-qat-Q4_K_M.gguf:   0%|          | 0.00/806M [00:00<?, ?B/s]

  ✅ QAT Q4_K_M: google_gemma-3-1b-it-qat-Q4_K_M.gguf (806 MB)


google_gemma-3-1b-it-qat-Q3_K_M.gguf:   0%|          | 0.00/722M [00:00<?, ?B/s]

  ✅ QAT Q3_K_M: google_gemma-3-1b-it-qat-Q3_K_M.gguf (722 MB)


...


  [  greeting] Char:89% Word:87% Rep:0.00
    Q: வணக்கம்
    A: வணக்கம்! நான் VAZHI, தமிழ்நாட்டு மக்களுக்கான AI உதவியாளர். உங்களுக்கு என்ன உதவி வேண்டும்? என்னிடம் என்ன கேட்க விரும்புகிறீர்கள்?

  [  identity] Char:96% Word:96% Rep:0.00
    Q: நீங்கள் யார்?
    A: நான் ஒரு பெரிய மொழி மாதிரி, கூகிள் நிறுவனத்தால் பயிற்சி அளிக்கப்பட்டது. நான் தமிழ் மொழியில் பதிலளிக்கும் திறன் கொண்ட ஒரு AI உதவியாளர். உங்களுக்கு என்ன உதவி வேண்டும் என்று சொல்லுங்கள்.

  [    health] Char:89% Word:91% Rep:0.05
    Q: காலையில் என்ன சாப்பிடலாம்?
    A: காலையில் சாப்பிட பல விஷயங்கள் உள்ளன. உங்களுக்கு எந்த மாதிரியான உணவு хочется? சில விருப்பங்கள் கீழே:

*   **சாதம் மற்றும் பழங்கள்:** சாதம் மற்றும் பழங்கள் ஒரு எளிய மற்றும் ஆரோக்கியமான காலை உணவு.
*   **பிரியாணி:** பிரியாணி ஒரு பிரபலமான மற்றும் சுவையான காலை உணவு.
*   **சப்பாத்தி மற்றும் தேன்:** சப்பாத்

  [      govt] Char:96% Word:92% Rep:0.08
    Q: ரேஷன் கார்டு பற்றி தகவல் தேவை
    A: ரேஷன் கார்டு பற்றி உங்களுக்கு என்ன தகவல் தேவை? நீங்கள் எந்த வகையான ரேஷன் கார்டு

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



  [  greeting] Char:91% Word:94% Rep:0.00
    Q: வணக்கம்
    A: வணக்கம்! வழி (VAZHI) என்றே நீங்கள் என்னை அழைத்தால், தமிழ்நாட்டில் உங்களுக்கு உதவ நான் தயார். நீங்கள் என்ன தெரிந்து கொள்ள விரும்புகிறீர்கள்?

  [  identity] Char:97% Word:100% Rep:0.00
    Q: நீங்கள் யார்?
    A: நான் ஒரு பெரிய மொழி மாதிரி. கூகிள் நிறுவனத்தால் பயிற்சி பெற்றது. நான் உங்களுக்கு எப்படி உதவ முடியும்? நீங்கள் என்ன தெரிந்து கொள்ள விரும்புகிறீர்கள்?

  [    health] Char:88% Word:90% Rep:0.08
    Q: காலையில் என்ன சாப்பிடலாம்?
    A: காலையில் சாப்பிட பல விருப்பங்கள் உள்ளன. நீங்கள் எதைச் சாப்பிட விரும்புகிறீர்கள் என்பதைப் பொறுத்து, சில விருப்பங்கள் இங்கே:

*   **சாதம்:** இது ஒரு பாரம்பரிய உணவு.
*   **பழங்கள்:** பழங்கள் உடலுக்கு ஆரோக்கியம் தரும்.
*   **சப்பாத்தி:** இது ஒரு பிரபலமான இந்திய உணவு.
*   **சாதா:** இது ஒரு பிரபலமான பானம்

  [      govt] Char:96% Word:89% Rep:0.06
    Q: ரேஷன் கார்டு பற்றி தகவல் தேவை
    A: ரேஷன் கார்டு பற்றி நீங்கள் என்ன தெரிந்து கொள்ள விரும்புகிறீர்கள்? உங்களுக்கு என்ன கேள்வி வேண்டும்?

ரே

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



  [  greeting] Char:95% Word:94% Rep:0.00
    Q: வணக்கம்
    A: வணக்கம்! நான் தமிழ்நாட்டு மக்களுக்கான AI உதவியாளர். உங்களுக்கு என்ன உதவி வேண்டும் என்று சொல்லுங்கள். நான் உங்களுக்கு எப்படி உதவ முடியும்?

  [  identity] Char:98% Word:100% Rep:0.00
    Q: நீங்கள் யார்?
    A: நான் ஒரு பெரிய மொழி மாதிரி. கூகிள் நிறுவனத்தால் பயிற்சி அளிக்கப்பட்டு, தமிழ் மொழியைப் புரிந்து கொண்டு பதிலளிக்க வடிவமைக்கப்பட்டுள்ளேன். நான் ஒரு மனிதனைப் போல பதில் அளிக்க முடியாது, ஆனால் உங்கள் கேள்விகளுக்குத் துல்லியமான மற்றும் தெளிவான பதில்களை வழங்குகிறேன். உங்களுக்கு எப்படி உதவ முடியும்?

  [    health] Char:90% Word:92% Rep:0.00
    Q: காலையில் என்ன சாப்பிடலாம்?
    A: காலையில் சாப்பிட நிறைய விருப்பங்கள் இருக்கு, என்ன உங்களுக்கு விருப்பம்? உங்க விருப்பம் என்னவென்றால், என்ன மாதிரியான உணவு உங்களுக்கு பிடிக்கும்?

*   **சாதம்:** சாதம், சாதம் மற்றும் பச்சைக் கஞ்சி மாதிரி ஏதாவது சாப்பிடலாம்.
*   **இட்லி:** இட்லி, சாம்பூலம், மற்றும் நெய் சேர்த்து சாப்பிட்டு சாப்பிடலாம்.

  [      govt] Char:92% Word:94% Rep:0.01
    Q

---

# Part A: imatrix Quantization + Embed/Output Tensor Strategy

**Goal:** Three experiments to improve Tamil quality and potentially reduce file size:
1. **imatrix:** Tamil-aware importance matrix to preserve Tamil-critical weights during quantization
2. **Embed/output quant:** Force embedding + lm_head tensors to Q8_0 (tests whether the ~600 MiB "non-shrinking floor" from Phase 26 was due to default high-precision tensor storage)
3. **Combined:** imatrix + embed/output quant together for maximum benefit

**Why imatrix:** Tells the quantizer which weights matter most for Tamil text. Can improve
quality at aggressive quant levels (Q2_K, IQ2_M) by preserving Tamil-critical weights at higher
precision. Does NOT significantly reduce file size.

**Why embed/output quant:** Phase 26 showed only ~110 MiB savings from Q4_K_M→Q2_K, suggesting
most of the model size is in tensors that don't shrink with standard quantization. llama.cpp
supports `--output-tensor-type` and `--token-embedding-type` flags that can explicitly quantize
the embedding and output head tensors (normally stored at higher precision).

**Quantization variants generated (4 categories):**
1. Baseline: Q4_K_M, Q3_K_M, Q2_K, IQ2_M (default precision)
2. imatrix: same levels with Tamil-aware importance matrix
3. Embed/output Q8_0: Q4_K_M, Q3_K_M, Q2_K with forced embed+output quantization
4. Combined: imatrix + embed/output Q8_0

**Steps:**
1. Build Tamil calibration corpus (~70% Tamil, ~30% English)
2. Generate importance matrix using `llama-imatrix`
3. Quantize all variants (16 total GGUF files)
4. Compare Tamil quality + file sizes across all variants
5. GO/NO-GO verdict per category

In [11]:
# Cell 5 — Build Tamil Calibration Corpus
#
# imatrix needs a representative text file to compute weight importance.
# We use ~70% Tamil + ~30% English to ensure the matrix captures:
#   - Tamil language patterns (Sadhguru articles, classical lit, SFT data)
#   - English reasoning ability (bartowski's standard calibration data)
#
# Target: ~50K tokens of mixed Tamil/English text.

from huggingface_hub import hf_hub_download
from datasets import load_dataset

tamil_texts = []
english_texts = []

# --- Source 1: Sadhguru Tamil articles (~562 articles, ~3.9M chars) ---
print("Loading Sadhguru articles...")
sadhguru_path = hf_hub_download(
    repo_id="CryptoYogi/vazhi-tamil-sft-v7_0",
    filename="vazhi-tamil-sft-v7_0-full.json",
    repo_type="dataset",
)
sft_data = json.load(open(sadhguru_path))
# Extract instruction + output text from SFT dataset (Tamil content)
for item in sft_data:
    text = item.get("instruction", "") + "\n" + item.get("output", "")
    text = text.strip()
    if text and tamil_char_pct(text) > 50:
        tamil_texts.append(text)
print(f"  SFT v7.0 Tamil samples: {len(tamil_texts)}")

# Also try loading Sadhguru raw articles if available locally
sadhguru_local = "data/sources/sft/sadhguru-raw/articles_filtered_full.json"
if os.path.exists(sadhguru_local):
    articles = json.load(open(sadhguru_local))
    for a in articles[:100]:  # Top 100 articles (diverse content)
        text = a.get("tamil_text", "")
        if len(text) > 500:
            # Take first 2000 chars from each article (avoid over-representing long ones)
            tamil_texts.append(text[:2000])
    print(f"  Added {min(len(articles), 100)} Sadhguru article excerpts")

# --- Source 2: Classical Tamil literature (DAPT corpus files) ---
print("Loading classical Tamil literature...")
dapt_dir = "data/sources/dapt"
classical_count = 0

if os.path.exists(dapt_dir):
    # Silapathikaram
    sil = json.load(open(f"{dapt_dir}/36_silapathikaram_corpus.json"))
    for item in sil:
        tamil_texts.append(item.get("text", ""))
    classical_count += len(sil)

    # Thirukkural (verse + meaning)
    kural = json.load(open(f"{dapt_dir}/37_thirukkural_corpus.json"))
    for item in kural:
        text = item.get("tamil", "") + " " + item.get("meaning_tamil", "")
        tamil_texts.append(text.strip())
    classical_count += len(kural)

    # Sangam literature
    sangam = json.load(open(f"{dapt_dir}/38_sangam_corpus.json"))
    for item in sangam:
        tamil_texts.append(item.get("text", ""))
    classical_count += len(sangam)

    # Bharathiar poetry
    bhar = json.load(open(f"{dapt_dir}/40_bharathiar_corpus.json"))
    for poem in bhar.get("poems", []):
        tamil_texts.append(poem.get("full_text", ""))
    classical_count += len(bhar.get("poems", []))

    # Aathichoodi
    aathi = json.load(open(f"{dapt_dir}/39_aathichoodi_corpus.json"))
    for key in ["aathichoodi", "konrai_venthan"]:
        if key in aathi and isinstance(aathi[key], list):
            for item in aathi[key]:
                text = item.get("tamil", "") if isinstance(item, dict) else str(item)
                tamil_texts.append(text)
    classical_count += sum(len(aathi.get(k, [])) for k in ["aathichoodi", "konrai_venthan"])

    print(f"  Classical literature items: {classical_count}")
else:
    print(f"  ⚠️ DAPT dir not found at {dapt_dir} — skipping classical literature")
    print(f"     (Upload from project repo or use SFT data alone)")

# --- Source 3: English calibration data (bartowski's standard) ---
# Download a small general English calibration file
print("Downloading English calibration data...")
try:
    en_cal_path = hf_hub_download(
        repo_id="bartowski/calibration_data",
        filename="calibration_data.txt",
        repo_type="dataset",
    )
    with open(en_cal_path, "r") as f:
        en_text = f.read()
    # Take chunks of ~2000 chars
    for i in range(0, len(en_text), 2000):
        chunk = en_text[i:i+2000].strip()
        if len(chunk) > 100:
            english_texts.append(chunk)
    print(f"  English calibration chunks: {len(english_texts)}")
except Exception as e:
    print(f"  ⚠️ bartowski calibration download failed: {e}")
    print(f"     Using SFT English prompts as fallback...")
    # Fallback: generate some English text from our eval prompts
    for p in ENGLISH_PROMPTS:
        english_texts.append(f"Question: {p['text']}\nAnswer: This is a question about {p['desc']}.")

# --- Combine: ~70% Tamil + ~30% English ---
print(f"\n--- Combining calibration corpus ---")

# Shuffle Tamil texts and take a subset to control size
import random
random.seed(42)
random.shuffle(tamil_texts)
random.shuffle(english_texts)

# Target ~50K tokens. Gemma 3 tokenizer: ~1 token/Tamil char, ~1 token/4 English chars
# So target ~35K Tamil chars + ~60K English chars ≈ 50K tokens
tamil_combined = "\n\n".join(tamil_texts)
english_combined = "\n\n".join(english_texts)

# Truncate to target sizes
MAX_TAMIL_CHARS = 200_000   # ~200K Tamil chars
MAX_ENGLISH_CHARS = 80_000  # ~80K English chars (20-30K tokens)

tamil_combined = tamil_combined[:MAX_TAMIL_CHARS]
english_combined = english_combined[:MAX_ENGLISH_CHARS]

# Interleave Tamil and English blocks
calibration_text = tamil_combined + "\n\n" + english_combined

CAL_FILE = "vazhi_calibration.txt"
with open(CAL_FILE, "w", encoding="utf-8") as f:
    f.write(calibration_text)

total_chars = len(calibration_text)
tamil_pct = tamil_char_pct(calibration_text)
print(f"\n✅ Calibration corpus saved: {CAL_FILE}")
print(f"   Total chars: {total_chars:,}")
print(f"   Tamil char%: {tamil_pct:.1f}%")
print(f"   File size:   {os.path.getsize(CAL_FILE) / 1e6:.1f} MB")

Loading Sadhguru articles...


vazhi-tamil-sft-v7_0-full.json: 0.00B [00:00, ?B/s]

  SFT v7.0 Tamil samples: 3714
Loading classical Tamil literature...
  ⚠️ DAPT dir not found at data/sources/dapt — skipping classical literature
     (Upload from project repo or use SFT data alone)
  ⚠️ bartowski calibration download failed: 404 Client Error. (Request ID: Root=1-699502cd-11975cc279274ad67ea489db;0c6a8525-292f-40a4-a580-127ac16b6672)

Repository Not Found for url: https://huggingface.co/datasets/bartowski/calibration_data/resolve/main/calibration_data.txt.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
     Using SFT English prompts as fallback...

--- Combining calibration corpus ---

✅ Calibration corpus saved: vazhi_calibration.txt
   Total chars: 200,263
   Tamil char%: 82.2%
   File size:   0.5 MB


In [12]:
# Cell 6 — Generate imatrix
#
# llama-imatrix processes the calibration corpus through the model and records
# which weights are most important for predicting the next token.
# --process-output includes the output/lm_head layer (critical for Tamil vocab).
#
# Runtime: ~10-15 min on L4 GPU, ~30 min on CPU

IMATRIX_FILE = "vazhi-imatrix.dat"

ngl = "99" if torch.cuda.is_available() else "0"

print(f"Generating importance matrix...")
print(f"  Model: {F16_GGUF}")
print(f"  Calibration: {CAL_FILE}")
print(f"  GPU layers: {ngl}")
print(f"  Output: {IMATRIX_FILE}")
print()

!./llama.cpp/build/bin/llama-imatrix \
    -m {F16_GGUF} \
    -f {CAL_FILE} \
    -o {IMATRIX_FILE} \
    --process-output \
    -ngl {ngl} \
    2>&1 | tail -20

if os.path.exists(IMATRIX_FILE):
    size = os.path.getsize(IMATRIX_FILE)
    print(f"\n✅ imatrix saved: {IMATRIX_FILE} ({size / 1e6:.1f} MB)")
else:
    print(f"\n❌ imatrix generation failed — check logs above")

Generating importance matrix...
  Model: vazhi-v7.1-f16.gguf
  Calibration: vazhi_calibration.txt
  GPU layers: 99
  Output: vazhi-imatrix.dat

[89]37.1597,[90]37.3021,[91]37.5074,[92]37.4725,[93]37.2991,[94]37.5843,[95]37.7931,[96]37.4392,
save_imatrix: saving imatrix using GGUF format with a different suffix than .gguf
save_imatrix: if you want the previous imatrix format, use --output-format dat

[97]37.6179,[98]37.7094,[99]37.9025,[100]37.9555,[101]38.1267,[102]38.1790,[103]37.9361,[104]37.6463,[105]38.2460,[106]38.2487,[107]38.2752,[108]38.3522,
save_imatrix: saving imatrix using GGUF format with a different suffix than .gguf
save_imatrix: if you want the previous imatrix format, use --output-format dat

[109]38.1257,[110]38.0988,[111]37.9289,[112]37.9954,
Final estimate: PPL = 37.9954 +/- 0.77567

save_imatrix: saving imatrix using GGUF format with a different suffix than .gguf
save_imatrix: if you want the previous imatrix format, use --output-format dat


llama_perf_context_pri

In [13]:
# Cell 7 — Quantize: Baselines + imatrix + embed/output quant variants
#
# Generate three categories of variants for fair comparison:
#   1. Baseline (no imatrix, default embed/output precision)
#   2. imatrix (Tamil-aware weight importance)
#   3. Embed/output quant (explicitly quantize the "non-shrinking" tensors)
#
# The embed/output quant tests whether the ~600 MiB "floor" seen in Phase 26
# can actually be lowered by forcing embed_tokens and lm_head to Q8_0.
# llama.cpp supports --output-tensor-type and --token-embedding-type flags.

QUANT_LEVELS = ["Q4_K_M", "Q3_K_M", "Q2_K", "IQ2_M"]

def quantize(f16_gguf, output_gguf, quant_type, imatrix_file=None,
             output_tensor_type=None, token_embedding_type=None):
    """Run llama-quantize with optional imatrix and embed/output type overrides."""
    cmd = f"./llama.cpp/build/bin/llama-quantize"
    if imatrix_file:
        cmd += f" --imatrix {imatrix_file}"
    if output_tensor_type:
        cmd += f" --output-tensor-type {output_tensor_type}"
    if token_embedding_type:
        cmd += f" --token-embedding-type {token_embedding_type}"
    cmd += f" {f16_gguf} {output_gguf} {quant_type}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
    if result.returncode != 0:
        print(f"  ❌ FAILED: {result.stderr[-500:]}")
        return False
    return True

gguf_files = {}  # label -> filepath

# --- Category 1: Baseline (no imatrix, default precision) ---
print("=" * 65)
print("Category 1: BASELINES (no imatrix, default embed/output)")
print("=" * 65)

for qt in QUANT_LEVELS:
    outfile = f"vazhi-v7.1-{qt.lower()}.gguf"
    print(f"\n  {qt} → {outfile}...", end=" ")
    if os.path.exists(outfile):
        print(f"exists ({os.path.getsize(outfile)/1e6:.0f} MB)")
    elif quantize(F16_GGUF, outfile, qt.lower()):
        print(f"✅ ({os.path.getsize(outfile)/1e6:.0f} MB)")
    else:
        print("❌")
        continue
    gguf_files[f"{qt} (baseline)"] = outfile

# --- Category 2: imatrix (Tamil-aware weight importance) ---
print("\n" + "=" * 65)
print("Category 2: IMATRIX (Tamil-aware)")
print("=" * 65)

for qt in QUANT_LEVELS:
    outfile = f"vazhi-v7.1-{qt.lower()}-imat.gguf"
    print(f"\n  {qt}+imatrix → {outfile}...", end=" ")
    if os.path.exists(outfile):
        print(f"exists ({os.path.getsize(outfile)/1e6:.0f} MB)")
    elif quantize(F16_GGUF, outfile, qt.lower(), IMATRIX_FILE):
        print(f"✅ ({os.path.getsize(outfile)/1e6:.0f} MB)")
    else:
        print("❌")
        continue
    gguf_files[f"{qt} (imatrix)"] = outfile

# --- Category 3: Embed/output quant (force embeddings + lm_head to Q8_0) ---
# This tests whether the "non-shrinking floor" from Phase 26 can be lowered.
# Gemma 3 1B has 262K×1536 = ~402M params in embeddings (f32/f16 by default).
# Forcing to Q8_0 could save ~200-300 MB if these tensors were at f16.
print("\n" + "=" * 65)
print("Category 3: EMBED/OUTPUT QUANT (force embed+output to Q8_0)")
print("=" * 65)

for qt in ["Q4_K_M", "Q3_K_M", "Q2_K"]:
    outfile = f"vazhi-v7.1-{qt.lower()}-eq8.gguf"
    print(f"\n  {qt}+embed_q8 → {outfile}...", end=" ")
    if os.path.exists(outfile):
        print(f"exists ({os.path.getsize(outfile)/1e6:.0f} MB)")
    elif quantize(F16_GGUF, outfile, qt.lower(),
                  output_tensor_type="q8_0", token_embedding_type="q8_0"):
        print(f"✅ ({os.path.getsize(outfile)/1e6:.0f} MB)")
    else:
        print("❌")
        continue
    gguf_files[f"{qt} (embed_q8)"] = outfile

# --- Category 4: imatrix + embed/output quant (best of both) ---
print("\n" + "=" * 65)
print("Category 4: IMATRIX + EMBED/OUTPUT QUANT (combined)")
print("=" * 65)

for qt in ["Q4_K_M", "Q3_K_M", "Q2_K"]:
    outfile = f"vazhi-v7.1-{qt.lower()}-imat-eq8.gguf"
    print(f"\n  {qt}+imatrix+embed_q8 → {outfile}...", end=" ")
    if os.path.exists(outfile):
        print(f"exists ({os.path.getsize(outfile)/1e6:.0f} MB)")
    elif quantize(F16_GGUF, outfile, qt.lower(), IMATRIX_FILE,
                  output_tensor_type="q8_0", token_embedding_type="q8_0"):
        print(f"✅ ({os.path.getsize(outfile)/1e6:.0f} MB)")
    else:
        print("❌")
        continue
    gguf_files[f"{qt} (imat+eq8)"] = outfile

# --- Size comparison table ---
print(f"\n\n{'='*75}")
print(f"FILE SIZE COMPARISON — All Variants")
print(f"{'='*75}")
print(f"  {'Variant':<30} {'Size (MB)':>10} {'Size (MiB)':>11} {'vs Baseline':>12}")
print(f"  {'─'*30} {'─'*10} {'─'*11} {'─'*12}")

# Get baseline sizes for delta calculation
baseline_sizes = {}
for qt in QUANT_LEVELS:
    key = f"{qt} (baseline)"
    if key in gguf_files:
        baseline_sizes[qt] = os.path.getsize(gguf_files[key])

for label, path in sorted(gguf_files.items()):
    size_bytes = os.path.getsize(path)
    mb = size_bytes / 1e6
    mib = size_bytes / 1048576
    # Find matching baseline for delta
    qt_match = label.split(" ")[0]
    bl_size = baseline_sizes.get(qt_match, size_bytes)
    delta = size_bytes - bl_size
    delta_str = f"{delta/1e6:+.0f} MB" if delta != 0 else "—"
    print(f"  {label:<30} {mb:>10.1f} {mib:>10.1f} {delta_str:>12}")

print(f"\n  f16 reference: {os.path.getsize(F16_GGUF)/1e6:.1f} MB")
print(f"\n  Key insight: if embed_q8 variants are significantly smaller,")
print(f"  the 'non-shrinking floor' from Phase 26 was due to default tensor precision.")

Category 1: BASELINES (no imatrix, default embed/output)

  Q4_K_M → vazhi-v7.1-q4_k_m.gguf... ✅ (806 MB)

  Q3_K_M → vazhi-v7.1-q3_k_m.gguf... ✅ (722 MB)

  Q2_K → vazhi-v7.1-q2_k.gguf... ✅ (690 MB)

  IQ2_M → vazhi-v7.1-iq2_m.gguf...   ❌ FAILED: ight - [ 6912,  1152,     1,     1], type =    f16, 

Missing importance matrix for tensor blk.3.ffn_down.weight in a very low-bit quantization
The result will be garbage, so bailing out

llama_model_quantize: failed to quantize: Missing importance matrix for tensor blk.3.ffn_down.weight in a very low-bit quantization
main: failed to quantize model from 'vazhi-v7.1-f16.gguf'

❌

Category 2: IMATRIX (Tamil-aware)

  Q4_K_M+imatrix → vazhi-v7.1-q4_k_m-imat.gguf... ✅ (806 MB)

  Q3_K_M+imatrix → vazhi-v7.1-q3_k_m-imat.gguf... ✅ (722 MB)

  Q2_K+imatrix → vazhi-v7.1-q2_k-imat.gguf... ✅ (690 MB)

  IQ2_M+imatrix → vazhi-v7.1-iq2_m-imat.gguf... ✅ (670 MB)

Category 3: EMBED/OUTPUT QUANT (force embed+output to Q8_0)

  Q4_K_M+embed_q8 → vazhi-v7.1-q

In [14]:
# Cell 8 — Tamil Quality Comparison: All Variants
#
# Run eval_gguf on each variant and collect results for comparison.
# This takes ~2-5 min per variant (13 prompts each).
# With 4 categories x 3-4 quant levels = ~15 variants, total ~30-75 min.

all_eval_results = {}

# Evaluate each GGUF variant
total = len(gguf_files)
for idx, (label, path) in enumerate(sorted(gguf_files.items())):
    print(f"\n[{idx+1}/{total}] Evaluating {label}...")
    result = eval_gguf(path, label)
    all_eval_results[label] = result

# --- Summary table ---
print(f"\n\n{'='*95}")
print(f"  QUALITY COMPARISON — All Quantization Variants")
print(f"{'='*95}")
print(f"\n  {'Variant':<30} {'Size(MB)':>8} {'TamilC%':>8} {'TamilW%':>8} {'Repeat':>7} {'NonEmpty':>9}")
print(f"  {'─'*30} {'─'*8} {'─'*8} {'─'*8} {'─'*7} {'─'*9}")

for label in sorted(all_eval_results.keys()):
    r = all_eval_results[label]
    ne_str = f"{r['non_empty']}/{len(r['results'])}"
    print(f"  {label:<30} {r['size_mb']:>8.1f} {r['avg_char']:>7.1f}% {r['avg_word']:>7.1f}% {r['avg_rep']:>6.2f} {ne_str:>9}")

# --- Per-prompt comparison for most impactful variants ---
print(f"\n\n{'─'*95}")
print(f"  PER-PROMPT: Q2_K baseline vs Q2_K imatrix vs Q2_K embed_q8")
print(f"{'─'*95}")

q2_bl = all_eval_results.get("Q2_K (baseline)", {}).get("results", [])
q2_im = all_eval_results.get("Q2_K (imatrix)", {}).get("results", [])
q2_eq = all_eval_results.get("Q2_K (embed_q8)", {}).get("results", [])

if q2_bl:
    for i in range(len(q2_bl)):
        b = q2_bl[i]
        print(f"\n  Q: {b['prompt']}")
        print(f"  baseline (C:{b['tamil_char_pct']:.0f}% W:{b['tamil_word_pct']:.0f}%): {b['response'][:150]}")
        if i < len(q2_im):
            im = q2_im[i]
            print(f"  imatrix  (C:{im['tamil_char_pct']:.0f}% W:{im['tamil_word_pct']:.0f}%): {im['response'][:150]}")
        if i < len(q2_eq):
            eq = q2_eq[i]
            print(f"  embed_q8 (C:{eq['tamil_char_pct']:.0f}% W:{eq['tamil_word_pct']:.0f}%): {eq['response'][:150]}")
else:
    print("  (Q2_K variants not available)")


[1/13] Evaluating IQ2_M (imatrix)...

  📊 EVALUATING: IQ2_M (imatrix)
  GGUF: vazhi-v7.1-iq2_m-imat.gguf
  Size: 669.8 MB


llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



  [  greeting] Char:95% Word:100% Rep:0.00
    Q: வணக்கம்
    A: எப்படிச் சொல்வீர்கள்?

  [  identity] Char:98% Word:100% Rep:0.00
    Q: நீங்கள் யார்?
    A: நான் கூகிள் மூலம் பயிற்சி பெற்ற ஒரு பெரிய மொழி மாதிரி. நான் கூகிள் தீருமிங்கில் உருவாக்கப்பட்டேன்.

  [    health] Char:84% Word:88% Rep:0.28
    Q: காலையில் என்ன சாப்பிடலாம்?
    A: காலையில் என்ன சாப்பிடலாம்னு யோசிச்சுக்கிட்டு இருக்கேன். இதோ சில யோசனைகள்:

*   **தக்காளி:** இது ஒரு நல்ல உணவு.
*   **பருப்பு:** இது ஒரு நல்ல உணவு.
*   **பழங்கள்:** இது ஒரு நல்ல உணவு.
*   **சாத:** இது ஒரு நல்ல உணவு.
*   **தமிழ்கள்:** இது ஒரு நல்ல உணவு.

உங்களிடம் என்ன சாப்பிடத் நினைக்கிறீர்கள்?

  [      govt] Char:96% Word:93% Rep:0.02
    Q: ரேஷன் கார்டு பற்றி தகவல் தேவை
    A: ரேஷன் கார்டு பற்றிய தகவல்கள் கீழே:

*   ரேஷன் கார்டு என்பது ரேஷன் பொருட்களின் (உணவு, எரிபொருள், கட்டணங்கள்) சரிபார்ப்புக்கான கருவி.
*   ரேஷன் கார்டு, ரேஷன் நிறுவனத்தின் பயன்பாட்டிற்காகப் பயன்படுத்தப்படும் ஒரும்.
*   இது ஒரு கணக்கு மற்றும் ரேஷன் நிறுவனத்தின் மூலம் உள்ள ஒரு தன

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



  [  greeting] Char:96% Word:100% Rep:0.00
    Q: வணக்கம்
    A: வணக்கம்! நீங்க என்ன உதவி பண்ணனும்ங்க? நான் தமிழ்ல பதில் சொல்ல தயாரா இருக்கேன்.

  [  identity] Char:97% Word:100% Rep:0.00
    Q: நீங்கள் யார்?
    A: நான் ஒரு பெரிய மொழி மாதிரி. கூகிள் மூலம் பயிற்சி அளிக்கப்பட்டது. நான் உங்களுக்கு எப்படி உதவு செய்வது என்று தெரியவில்லை.

  [    health] Char:83% Word:87% Rep:0.30
    Q: காலையில் என்ன சாப்பிடலாம்?
    A: காலையில் சாப்பிட சில விஷயங்கள் இங்கே:

*   **சாதம்:** இது ஒரு எளிதான உணவு.
*   **சாதம் மற்றும் சர்க்கரை:** இது ஒரு சுறுப்பான உணவு.
*   **உணவு:** இது ஒரு சுறுப்பான உணவு.
*   **அரிசி:** இது ஒரு சுறுப்பான உணவு.
*   **குளூட்:** இது ஒரு சுறுப்பான உணவு.
*   **அலங்காரம்:** இது ஒரு சுறுப்பான உணவு.
*   **காய

  [      govt] Char:95% Word:100% Rep:0.00
    Q: ரேஷன் கார்டு பற்றி தகவல் தேவை
    A: சாரி, நான் தமிழ் மொழி பேசினால் எனக்கு உதவ முடியாது.

  [   culture] Char:97% Word:100% Rep:0.00
    Q: திருக்குறள் பற்றி சொல்லுங்கள்
    A: சரியாக! திருக்குறள் பற்றி நீங்கள் என்ன தெரிந்து கொ

...


  [   english] Char:0% Word:0% Rep:0.00
    Q: What is your name?
    A: [ERROR: Failed to create llama_context]


llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



  [   english] Char:0% Word:0% Rep:0.00
    Q: Tell me about Tamil Nadu
    A: [ERROR: Failed to create llama_context]


llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



  [   english] Char:0% Word:0% Rep:0.00
    Q: How do I apply for a passport?
    A: [ERROR: Failed to create llama_context]

  ──────────────────────────────────────────────────
  📊 Q4_K_M (imatrix) SUMMARY (Tamil prompts only):
     Avg Tamil char: 0.0%
     Avg Tamil word: 0.0%
     Avg repeat:     0.00
     Non-empty:      13/13
     Size:           806.1 MB


  QUALITY COMPARISON — All Quantization Variants

  Variant                        Size(MB)  TamilC%  TamilW%  Repeat  NonEmpty
  ────────────────────────────── ──────── ──────── ──────── ─────── ─────────
  IQ2_M (imatrix)                   669.8    93.8%    93.8%   0.18     13/13
  Q2_K (baseline)                   689.8    94.5%    97.5%   0.05     13/13
  Q2_K (embed_q8)                   689.8    94.5%    97.5%   0.05     13/13
  Q2_K (imat+eq8)                   689.8    94.5%    95.1%   0.04     13/13
  Q2_K (imatrix)                    689.8    94.5%    95.1%   0.04     13/13
  Q3_K_M (baseline)                 722.4

In [15]:
# Cell 9 — Part A GO/NO-GO Verdict
#
# Analyze all three categories: baseline vs imatrix vs embed/output quant.

print("=" * 75)
print("  PART A VERDICT: imatrix + Embed/Output Quantization")
print("=" * 75)

# --- imatrix comparison ---
print(f"\n  1. imatrix Impact (Tamil quality at each quant level):")
print(f"  {'─'*65}")
for qt in QUANT_LEVELS:
    bl_key = f"{qt} (baseline)"
    im_key = f"{qt} (imatrix)"
    bl = all_eval_results.get(bl_key)
    im = all_eval_results.get(im_key)
    if bl and im:
        word_delta = im['avg_word'] - bl['avg_word']
        direction = "↑" if word_delta > 0 else ("↓" if word_delta < 0 else "→")
        print(f"    {qt}: {bl['avg_word']:.1f}% → {im['avg_word']:.1f}% word ({direction}{abs(word_delta):.1f}%)")

# --- Embed/output quant comparison ---
print(f"\n  2. Embed/Output Quant Impact (size reduction from forcing Q8_0):")
print(f"  {'─'*65}")
for qt in ["Q4_K_M", "Q3_K_M", "Q2_K"]:
    bl_key = f"{qt} (baseline)"
    eq_key = f"{qt} (embed_q8)"
    bl = all_eval_results.get(bl_key)
    eq = all_eval_results.get(eq_key)
    if bl and eq:
        size_delta = eq['size_mb'] - bl['size_mb']
        word_delta = eq['avg_word'] - bl['avg_word']
        print(f"    {qt}: {bl['size_mb']:.0f}→{eq['size_mb']:.0f} MB ({size_delta:+.0f} MB), quality {bl['avg_word']:.0f}→{eq['avg_word']:.0f}% word ({word_delta:+.1f}%)")

# --- Combined (imatrix + embed/output) ---
print(f"\n  3. Combined Impact (imatrix + embed Q8_0):")
print(f"  {'─'*65}")
for qt in ["Q4_K_M", "Q3_K_M", "Q2_K"]:
    bl_key = f"{qt} (baseline)"
    combo_key = f"{qt} (imat+eq8)"
    bl = all_eval_results.get(bl_key)
    combo = all_eval_results.get(combo_key)
    if bl and combo:
        size_delta = combo['size_mb'] - bl['size_mb']
        word_delta = combo['avg_word'] - bl['avg_word']
        print(f"    {qt}: {bl['size_mb']:.0f}→{combo['size_mb']:.0f} MB ({size_delta:+.0f} MB), quality {bl['avg_word']:.0f}→{combo['avg_word']:.0f}% word ({word_delta:+.1f}%)")

# --- Conclusions ---
print(f"\n\n  {'─'*65}")
print(f"  CONCLUSIONS:")
print(f"  {'─'*65}")
print(f"  imatrix:")
print(f"    → Does NOT reduce file size (expected)")
print(f"    → Check word% improvements above — if >5% at Q2_K, use for 6GB+ tier")
print(f"")
print(f"  Embed/output quant (Q8_0):")
print(f"    → Check size reductions above")
print(f"    → If >100 MB savings with <5% quality drop, this is a major win")
print(f"    → This directly addresses Phase 26's 'non-shrinking floor' observation")
print(f"")
print(f"  Combined (imatrix + embed_q8):")
print(f"    → Best quality + smallest size = ideal for 6GB+ default")
print(f"    → If small enough (<500 MB), may even work on some 4GB devices")
print(f"")
print(f"  NEXT:")
print(f"    → Best imatrix variant becomes 6GB+ default")
print(f"    → If embed_q8 Q2_K or combined variant < 500 MB with good Tamil,")
print(f"      test on 4GB device before committing to vocab trimming")
print(f"    → Otherwise proceed to Part B (vocab trimming)")

  PART A VERDICT: imatrix + Embed/Output Quantization

  1. imatrix Impact (Tamil quality at each quant level):
  ─────────────────────────────────────────────────────────────────
    Q4_K_M: 0.0% → 0.0% word (→0.0%)
    Q3_K_M: 0.0% → 0.0% word (→0.0%)
    Q2_K: 97.5% → 95.1% word (↓2.4%)

  2. Embed/Output Quant Impact (size reduction from forcing Q8_0):
  ─────────────────────────────────────────────────────────────────
    Q4_K_M: 806→806 MB (+0 MB), quality 0→0% word (+0.0%)
    Q3_K_M: 722→722 MB (+0 MB), quality 0→0% word (+0.0%)
    Q2_K: 690→690 MB (+0 MB), quality 97→97% word (+0.0%)

  3. Combined Impact (imatrix + embed Q8_0):
  ─────────────────────────────────────────────────────────────────
    Q4_K_M: 806→806 MB (+0 MB), quality 0→0% word (+0.0%)
    Q3_K_M: 722→722 MB (+0 MB), quality 0→0% word (+0.0%)
    Q2_K: 690→690 MB (+0 MB), quality 97→95% word (-2.4%)


  ─────────────────────────────────────────────────────────────────
  CONCLUSIONS:
  ────────────────────────

---

# Part B: Vocabulary Trimming

**Goal:** Trim Gemma 3's 262K vocabulary down to ~40-50K tokens, cutting the embedding floor
from ~576 MiB (F16) to ~110 MiB, enabling a ~300 MiB Q4_K_M GGUF that fits on 4GB devices.

**Why this works:**
- 30% of Gemma 3's ~1B params are in the 262K embedding matrix (f32/f16, unquantized)
- Most of those 262K tokens are never used for Tamil/English text
- Removing unused tokens shrinks embeddings proportionally
- Additional benefit: ~80% reduction in runtime logits buffer (262K→50K softmax)

**Research backing:**
- SqueezeBits (2025): tested on Gemma 3 1B-it specifically — 15-40% speedup, minimal quality loss
- Estonian language paper: ~33% vocab pruned with no observable negative effect, no recovery SFT needed

**Steps:**
1. Token frequency analysis (what tokens does our Tamil+English text actually use?)
2. Build keep-list (special tokens + Tamil tokens + used tokens + byte fallbacks)
3. Trim model tensors (embedding + lm_head)
4. Rebuild tokenizer (SentencePiece protobuf + HF tokenizer.json)
5. Quality check (HF model inference)
6. Optional recovery SFT (if quality degrades)
7. GGUF conversion + quantization
8. Final quality validation

In [16]:
# Cell 10 — Token Frequency Analysis
#
# Tokenize our Tamil + English corpora to find which of the 262K tokens
# are actually used. Tokens never seen in our data are safe to remove.
#
# Sources:
#   - DAPT v2.1 corpus (39.5M tokens, largest Tamil corpus)
#   - SFT v7.0 dataset (4,172 instruction/output pairs)
#   - Calibration text from Part A (Tamil lit + English)

from transformers import AutoTokenizer
from collections import Counter

print("Loading Gemma 3 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
VOCAB_SIZE = len(tokenizer)
print(f"  Vocab size: {VOCAB_SIZE:,}")

token_counts = Counter()

# --- Source 1: DAPT v2.1 corpus (39.5M tokens) ---
print("\nTokenizing DAPT v2.1 corpus (this may take a few minutes)...")
try:
    dapt_ds = load_dataset("CryptoYogi/vazhi-dapt-tamil-v2_1", split="train")
    batch_size = 1000
    for i in range(0, len(dapt_ds), batch_size):
        batch = dapt_ds[i:i+batch_size]
        # DAPT blocks have 'input_ids' (pre-tokenized) or 'text' field
        if "input_ids" in batch:
            for ids in batch["input_ids"]:
                token_counts.update(ids)
        elif "text" in batch:
            for text in batch["text"]:
                ids = tokenizer.encode(text, add_special_tokens=False)
                token_counts.update(ids)
        if i % 10000 == 0:
            print(f"  Processed {i:,}/{len(dapt_ds):,} blocks...")
    print(f"  ✅ DAPT v2.1: {len(dapt_ds):,} blocks tokenized")
except Exception as e:
    print(f"  ⚠️ DAPT v2.1 failed: {e}")
    print(f"     Falling back to calibration text only")

# --- Source 2: SFT v7.0 dataset ---
print("\nTokenizing SFT v7.0 dataset...")
try:
    sft_path = hf_hub_download(
        repo_id="CryptoYogi/vazhi-tamil-sft-v7_0",
        filename="vazhi-tamil-sft-v7_0-full.json",
        repo_type="dataset",
    )
    sft_data = json.load(open(sft_path))
    for item in sft_data:
        text = item.get("instruction", "") + " " + item.get("output", "")
        ids = tokenizer.encode(text.strip(), add_special_tokens=False)
        token_counts.update(ids)
    print(f"  ✅ SFT v7.0: {len(sft_data):,} samples tokenized")
except Exception as e:
    print(f"  ⚠️ SFT v7.0 failed: {e}")

# --- Source 3: Calibration text from Part A ---
print("\nTokenizing calibration text...")
if os.path.exists(CAL_FILE):
    with open(CAL_FILE, "r") as f:
        cal_text = f.read()
    ids = tokenizer.encode(cal_text, add_special_tokens=False)
    token_counts.update(ids)
    print(f"  ✅ Calibration: {len(ids):,} tokens")

# --- Source 4: Gemma chat template tokens ---
# Ensure chat tokens are counted even if not in corpus
chat_texts = [
    "<start_of_turn>user\nவணக்கம்<end_of_turn>\n<start_of_turn>model\n",
    "<start_of_turn>user\nHello<end_of_turn>\n<start_of_turn>model\n",
]
for ct in chat_texts:
    ids = tokenizer.encode(ct, add_special_tokens=True)
    token_counts.update(ids)

# --- Analysis ---
unique_tokens_seen = len(token_counts)
total_token_occurrences = sum(token_counts.values())

print(f"\n{'='*60}")
print(f"TOKEN FREQUENCY ANALYSIS")
print(f"{'='*60}")
print(f"  Total vocab:          {VOCAB_SIZE:,}")
print(f"  Unique tokens seen:   {unique_tokens_seen:,} ({100*unique_tokens_seen/VOCAB_SIZE:.1f}%)")
print(f"  Tokens NEVER seen:    {VOCAB_SIZE - unique_tokens_seen:,} ({100*(VOCAB_SIZE-unique_tokens_seen)/VOCAB_SIZE:.1f}%)")
print(f"  Total occurrences:    {total_token_occurrences:,}")

# Distribution by Unicode script
tamil_token_count = 0
cjk_token_count = 0
latin_token_count = 0
other_token_count = 0

for token_id in range(VOCAB_SIZE):
    piece = tokenizer.convert_ids_to_tokens(token_id)
    if piece is None:
        continue
    decoded = piece.replace("▁", " ").strip()
    if re.search(r'[\u0B80-\u0BFF]', decoded):
        tamil_token_count += 1
    elif re.search(r'[\u4E00-\u9FFF\u3040-\u309F\u30A0-\u30FF\uAC00-\uD7AF]', decoded):
        cjk_token_count += 1
    elif re.search(r'[a-zA-Z]', decoded):
        latin_token_count += 1
    else:
        other_token_count += 1

print(f"\n  Token distribution by script:")
print(f"    Tamil (U+0B80-0BFF):  {tamil_token_count:,}")
print(f"    Latin (a-zA-Z):       {latin_token_count:,}")
print(f"    CJK/Kana/Hangul:      {cjk_token_count:,}")
print(f"    Other/special:        {other_token_count:,}")

# Top 20 most frequent tokens
print(f"\n  Top 20 most frequent tokens:")
for tid, count in token_counts.most_common(20):
    piece = tokenizer.convert_ids_to_tokens(tid)
    print(f"    [{tid:>6}] {piece:<20} {count:>10,}")

Loading Gemma 3 tokenizer...


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/356 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/33.1M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/32.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/38580 [00:00<?, ? examples/s]

  Processed 0/38,580 blocks...
  Processed 10,000/38,580 blocks...
  Processed 20,000/38,580 blocks...
  Processed 30,000/38,580 blocks...
  ✅ DAPT v2.1: 38,580 blocks tokenized

Tokenizing SFT v7.0 dataset...
  ✅ SFT v7.0: 4,172 samples tokenized

Tokenizing calibration text...
  ✅ Calibration: 57,843 tokens

TOKEN FREQUENCY ANALYSIS
  Total vocab:          262,145
  Unique tokens seen:   14,497 (5.5%)
  Tokens NEVER seen:    247,648 (94.5%)
  Total occurrences:    40,118,564

  Token distribution by script:
    Tamil (U+0B80-0BFF):  3,850
    Latin (a-zA-Z):       158,266
    CJK/Kana/Hangul:      28,357
    Other/special:        71,672

  Top 20 most frequent tokens:
    [ 70597] ▁poste                3,658,789
    [ 63400] ▁laughs               3,449,773
    [ 83198] ▁coworkers            2,283,948
    [ 99012] 수는                    1,897,671
    [ 31501] מי                    1,877,653
    [ 46354] ంధ                    1,829,920
    [ 20026] acker                 1,680,443
    [ 

In [17]:
# Cell 11 — Build Keep-List
#
# Determine which token IDs to keep. We keep tokens from 5 categories:
#   1. Special tokens (BOS, EOS, PAD, UNK, chat tokens)
#   2. Byte fallback tokens (<0x00> through <0xFF>) — essential for unseen chars
#   3. All tokens containing Tamil Unicode characters (U+0B80-U+0BFF)
#   4. All tokens that appeared in our corpus (frequency >= 1)
#   5. Common English tokens for govt acronyms, URLs, numbers, punctuation

keep_ids = set()

# --- Category 1: Special tokens ---
special_tokens = set()

# Add all special token IDs from tokenizer config
for attr in ['bos_token_id', 'eos_token_id', 'pad_token_id', 'unk_token_id']:
    tid = getattr(tokenizer, attr, None)
    if tid is not None:
        special_tokens.add(tid)

# Add any additional special tokens
if hasattr(tokenizer, 'additional_special_tokens_ids'):
    special_tokens.update(tokenizer.additional_special_tokens_ids)

# Add all tokens from added_tokens_encoder (chat template tokens like <start_of_turn>)
if hasattr(tokenizer, 'added_tokens_encoder'):
    for token_str, tid in tokenizer.added_tokens_encoder.items():
        special_tokens.add(tid)

keep_ids.update(special_tokens)
print(f"Category 1 — Special tokens: {len(special_tokens)}")

# --- Category 2: Byte fallback tokens (<0x00> through <0xFF>) ---
byte_tokens = set()
for token_id in range(VOCAB_SIZE):
    piece = tokenizer.convert_ids_to_tokens(token_id)
    if piece and re.match(r'^<0x[0-9A-Fa-f]{2}>$', piece):
        byte_tokens.add(token_id)

keep_ids.update(byte_tokens)
print(f"Category 2 — Byte fallback tokens: {len(byte_tokens)}")

# --- Category 3: All tokens containing Tamil Unicode chars ---
tamil_tokens = set()
for token_id in range(VOCAB_SIZE):
    piece = tokenizer.convert_ids_to_tokens(token_id)
    if piece and re.search(r'[\u0B80-\u0BFF]', piece):
        tamil_tokens.add(token_id)

keep_ids.update(tamil_tokens)
print(f"Category 3 — Tamil Unicode tokens: {len(tamil_tokens)}")

# --- Category 4: Tokens seen in corpus (frequency >= 1) ---
corpus_tokens = set(token_counts.keys())
keep_ids.update(corpus_tokens)
print(f"Category 4 — Corpus-seen tokens: {len(corpus_tokens)}")

# --- Category 5: Common English tokens for safety margin ---
# Include common English words, digits, punctuation that might not appear
# in our corpus but are needed for government acronyms, URLs, etc.
safety_texts = [
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ abcdefghijklmnopqrstuvwxyz",
    "0123456789 . , ; : ! ? - _ / \\ @ # $ % & * ( ) [ ] { }",
    "http https www .com .org .in .gov RTI OBC SC ST MBC BC",
    "PM CM TN TNPSC NEET JEE UGC AICTE PhD MBA",
    "Rs INR USD crore lakh ₹",
    "COVID vaccine hospital doctor medicine pharmacy",
    "police FIR complaint court advocate lawyer",
    "bank loan EMI interest rate percent",
    "Aadhaar PAN passport voter ID card ration",
    "Chennai Madurai Coimbatore Trichy Salem Tirunelveli",
    "Tamil Nadu India Karnataka Kerala Andhra Pradesh",
]
for text in safety_texts:
    ids = tokenizer.encode(text, add_special_tokens=False)
    keep_ids.update(ids)
print(f"Category 5 — English safety tokens added")

# --- Final keep-list ---
keep_ids_sorted = sorted(keep_ids)
NEW_VOCAB_SIZE = len(keep_ids_sorted)
removed_count = VOCAB_SIZE - NEW_VOCAB_SIZE

# Build old_id -> new_id mapping
old_to_new = {old_id: new_id for new_id, old_id in enumerate(keep_ids_sorted)}

# Calculate projected embedding savings
HIDDEN_DIM = 1536  # Gemma 3 1B-it hidden dimension (check model config)
# Try to get from config
try:
    import json as _json
    cfg = _json.load(open(f"{LOCAL_MODEL_DIR}/config.json"))
    HIDDEN_DIM = cfg.get("hidden_size", 1536)
except:
    pass

orig_emb_params = VOCAB_SIZE * HIDDEN_DIM
new_emb_params = NEW_VOCAB_SIZE * HIDDEN_DIM
emb_saving_params = orig_emb_params - new_emb_params
emb_saving_mb_f16 = emb_saving_params * 2 / 1e6  # f16 = 2 bytes per param

print(f"\n{'='*60}")
print(f"KEEP-LIST SUMMARY")
print(f"{'='*60}")
print(f"  Original vocab:  {VOCAB_SIZE:,}")
print(f"  Kept tokens:     {NEW_VOCAB_SIZE:,} ({100*NEW_VOCAB_SIZE/VOCAB_SIZE:.1f}%)")
print(f"  Removed tokens:  {removed_count:,} ({100*removed_count/VOCAB_SIZE:.1f}%)")
print(f"  Hidden dim:      {HIDDEN_DIM}")
print(f"\n  Embedding savings (token_embd only):")
print(f"    Original: {orig_emb_params:,} params ({orig_emb_params*2/1e6:.0f} MB f16)")
print(f"    Trimmed:  {new_emb_params:,} params ({new_emb_params*2/1e6:.0f} MB f16)")
print(f"    Saved:    {emb_saving_params:,} params ({emb_saving_mb_f16:.0f} MB f16)")
print(f"\n  If lm_head is separate (untied): savings double to ~{2*emb_saving_mb_f16:.0f} MB f16")
print(f"\n  Projected Q4_K_M GGUF after trimming:")
print(f"    ~{762 - emb_saving_mb_f16:.0f} MB (if tied) or ~{762 - 2*emb_saving_mb_f16:.0f} MB (if untied)")

# Save keep-list for later use
import pickle
with open("keep_ids.pkl", "wb") as f:
    pickle.dump({"keep_ids_sorted": keep_ids_sorted, "old_to_new": old_to_new}, f)
print(f"\n  Keep-list saved to keep_ids.pkl")

Category 1 — Special tokens: 6415
Category 2 — Byte fallback tokens: 256
Category 3 — Tamil Unicode tokens: 3850
Category 4 — Corpus-seen tokens: 14497
Category 5 — English safety tokens added

KEEP-LIST SUMMARY
  Original vocab:  262,145
  Kept tokens:     21,068 (8.0%)
  Removed tokens:  241,077 (92.0%)
  Hidden dim:      1152

  Embedding savings (token_embd only):
    Original: 301,991,040 params (604 MB f16)
    Trimmed:  24,270,336 params (49 MB f16)
    Saved:    277,720,704 params (555 MB f16)

  If lm_head is separate (untied): savings double to ~1111 MB f16

  Projected Q4_K_M GGUF after trimming:
    ~207 MB (if tied) or ~-349 MB (if untied)

  Keep-list saved to keep_ids.pkl


In [19]:
  max_id = max(keep_ids)
  print(f"  Max token ID in keep_ids: {max_id}")
  print(f"  Vocab size: {model.config.vocab_size}")
  oob = [id for id in keep_ids if id >= model.config.vocab_size]
  print(f"  Out-of-bounds IDs: {oob}")


  Max token ID in keep_ids: 262144
  Vocab size: 262144
  Out-of-bounds IDs: [262144]


In [20]:
  # Filter out any IDs >= vocab_size (the embedding matrix is 0-indexed)
  vocab_size = model.config.vocab_size  # 262144
  keep_ids_sorted = sorted([id for id in keep_ids if id < vocab_size])
  print(f"  Filtered keep_ids: {len(keep_ids_sorted)} (removed {len(keep_ids) - len(keep_ids_sorted)} out-of-range)")

  keep_tensor = torch.tensor(keep_ids_sorted, dtype=torch.long)
  new_embed = embed_weight[keep_tensor].clone()


  Filtered keep_ids: 21067 (removed 1 out-of-range)


In [22]:
  keep_ids_sorted = sorted([id for id in keep_ids if id < vocab_size])
  NEW_VOCAB_SIZE = len(keep_ids_sorted)  # 21,067 — the real count


In [23]:
# Cell 12 — Trim Model Tensors
#
# Load v7.1 model, extract embedding rows for kept tokens only,
# update config, save trimmed model.
#
# Key: check tie_word_embeddings — if False, trim both embed_tokens AND lm_head.

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch, gc

TRIMMED_MODEL_DIR = "./vazhi-v7.1-trimmed"

print(f"Loading {HF_MODEL} in fp16 on CPU...")
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True,
)

config = model.config
tied = getattr(config, 'tie_word_embeddings', True)
orig_vocab = config.vocab_size

print(f"  Original vocab_size: {orig_vocab:,}")
print(f"  tie_word_embeddings: {tied}")
print(f"  New vocab_size:      {NEW_VOCAB_SIZE:,}")

# --- Locate embedding and lm_head tensors ---
# Gemma 3 architecture: model.embed_tokens.weight and model.lm_head.weight
embed_weight = model.model.embed_tokens.weight.data  # [vocab_size, hidden_dim]
print(f"\n  embed_tokens shape: {embed_weight.shape}")

if not tied:
    lm_head_weight = model.lm_head.weight.data  # [vocab_size, hidden_dim]
    print(f"  lm_head shape:      {lm_head_weight.shape}")
else:
    print(f"  lm_head: TIED (shares embed_tokens weight)")

# --- Extract kept rows ---
keep_tensor = torch.tensor(keep_ids_sorted, dtype=torch.long)

new_embed = embed_weight[keep_tensor].clone()  # [new_vocab, hidden_dim]
print(f"\n  New embed_tokens shape: {new_embed.shape}")

if not tied:
    new_lm_head = lm_head_weight[keep_tensor].clone()  # [new_vocab, hidden_dim]
    print(f"  New lm_head shape:      {new_lm_head.shape}")

# --- Resize model ---
# Use resize_token_embeddings to handle the internal bookkeeping
model.resize_token_embeddings(NEW_VOCAB_SIZE)

# Overwrite with our carefully selected rows (resize_token_embeddings
# initializes new rows randomly, but we want specific old rows)
model.model.embed_tokens.weight.data = new_embed

if not tied:
    model.lm_head.weight.data = new_lm_head

# Update config
model.config.vocab_size = NEW_VOCAB_SIZE

# Verify
assert model.model.embed_tokens.weight.shape[0] == NEW_VOCAB_SIZE
if not tied:
    assert model.lm_head.weight.shape[0] == NEW_VOCAB_SIZE

print(f"\n✅ Model trimmed: {orig_vocab:,} → {NEW_VOCAB_SIZE:,} vocab")

# --- Calculate actual parameter savings ---
orig_params = sum(p.numel() for p in AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR, torch_dtype=torch.float16, device_map="cpu",
    low_cpu_mem_usage=True
).parameters())
new_params = sum(p.numel() for p in model.parameters())
saved_params = orig_params - new_params

# Free the original model reload immediately
gc.collect()

print(f"  Original params: {orig_params:,}")
print(f"  Trimmed params:  {new_params:,}")
print(f"  Saved params:    {saved_params:,} ({100*saved_params/orig_params:.1f}%)")
print(f"  Saved (f16):     {saved_params*2/1e6:.0f} MB")

# --- Save trimmed model ---
print(f"\nSaving trimmed model to {TRIMMED_MODEL_DIR}...")
model.save_pretrained(TRIMMED_MODEL_DIR, safe_serialization=True)
print(f"✅ Trimmed model saved")

# Free memory
del model, new_embed
if not tied:
    del new_lm_head
gc.collect()
torch.cuda.empty_cache()

!ls -lh {TRIMMED_MODEL_DIR}/*.safetensors 2>/dev/null
!du -sh {TRIMMED_MODEL_DIR}

Loading CryptoYogi/vazhi-v7_1 in fp16 on CPU...
  Original vocab_size: 262,144
  tie_word_embeddings: True
  New vocab_size:      21,067

  embed_tokens shape: torch.Size([262144, 1152])
  lm_head: TIED (shares embed_tokens weight)

  New embed_tokens shape: torch.Size([21067, 1152])

✅ Model trimmed: 262,144 → 21,067 vocab
  Original params: 999,885,952
  Trimmed params:  722,165,248
  Saved params:    277,720,704 (27.8%)
  Saved (f16):     555 MB

Saving trimmed model to ./vazhi-v7.1-trimmed...
✅ Trimmed model saved
-rw-r--r-- 1 root root 1.4G Feb 18 01:51 ./vazhi-v7.1-trimmed/model.safetensors
1.4G	./vazhi-v7.1-trimmed


In [28]:
  print(f"Model type: {tok_json['model']['type']}")
  print(f"Merges count: {len(tok_json['model'].get('merges', []))}")
  print(f"First merge: {tok_json['model']['merges'][0] if tok_json['model'].get('merges') else 'N/A'}")
  print(f"Vocab type: {type(list(tok_json['model']['vocab'].values())[0]) if tok_json['model'].get('vocab') else 'N/A'}")

  #If the model type is "Unigram", the merges list might not even be relevant — the real issue is likely the added_tokens still referencing old 262K IDs. Check:

  max_added_id = max(t['id'] for t in tok_json['added_tokens'])
  print(f"Max added_token ID: {max_added_id}")
  print(f"New vocab size: {len(tok_json['model']['vocab'])}")


Model type: BPE
Merges count: 514906
First merge: ['\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n', '\n']
Vocab type: <class 'int'>
Max added_token ID: 21067
New vocab size: 21067


In [27]:
  import json

  with open('./vazhi-v7.1-trimmed/tokenizer.json', 'r') as f:
      tok_json = json.load(f)

  # Build set of valid token strings from the trimmed vocab
  valid_tokens = set(tok_json['model']['vocab'].keys())

  # Filter merges: keep only merges where both parts AND result are in vocab
  if 'merges' in tok_json['model']:
      original_merges = tok_json['model']['merges']
      filtered_merges = []
      for merge in original_merges:
          parts = merge.split(' ')
          if len(parts) == 2:
              # The merge produces a concatenation of the two parts
              merged = parts[0] + parts[1]
              if parts[0] in valid_tokens and parts[1] in valid_tokens and merged in valid_tokens:
                  filtered_merges.append(merge)
      tok_json['model']['merges'] = filtered_merges
      print(f"Merges: {len(original_merges)} → {len(filtered_merges)}")

  # Also filter added_tokens to only include tokens in the new vocab range
  if 'added_tokens' in tok_json:
      tok_json['added_tokens'] = [
          t for t in tok_json['added_tokens']
          if t['id'] < len(valid_tokens)
      ]
      print(f"Added tokens filtered to {len(tok_json['added_tokens'])}")

  with open('./vazhi-v7.1-trimmed/tokenizer.json', 'w') as f:
      json.dump(tok_json, f, ensure_ascii=False)


AttributeError: 'list' object has no attribute 'split'

In [29]:
  valid_tokens = set(tok_json['model']['vocab'].keys())

  # Filter merges — both parts AND merged result must be in trimmed vocab
  original_merges = tok_json['model']['merges']
  filtered_merges = []
  for merge in original_merges:
      a, b = merge[0], merge[1]
      merged = a + b
      if a in valid_tokens and b in valid_tokens and merged in valid_tokens:
          filtered_merges.append(merge)

  print(f"Merges: {len(original_merges)} → {len(filtered_merges)}")
  tok_json['model']['merges'] = filtered_merges

  # Fix added_tokens — drop any with id >= vocab_size
  NEW_VOCAB = len(tok_json['model']['vocab'])
  before = len(tok_json['added_tokens'])
  tok_json['added_tokens'] = [t for t in tok_json['added_tokens'] if t['id'] < NEW_VOCAB]
  print(f"Added tokens: {before} → {len(tok_json['added_tokens'])}")

  with open('./vazhi-v7.1-trimmed/tokenizer.json', 'w') as f:
      json.dump(tok_json, f, ensure_ascii=False)

  print("Saved. Now retry tokenizer load.")


Merges: 514906 → 18440
Added tokens: 6415 → 6414
Saved. Now retry tokenizer load.


In [30]:
# Cell 13 — Rebuild Tokenizer
#
# The trimmed model needs a matching tokenizer with only the kept tokens.
# Two files to rebuild:
#   1. tokenizer.model (SentencePiece protobuf) — the actual tokenizer logic
#   2. tokenizer.json (HuggingFace fast tokenizer) — used by transformers
#
# Strategy:
#   - Parse the original SentencePiece model protobuf
#   - Keep only pieces whose original index is in our keep-list
#   - Reindex to contiguous 0..N-1
#   - Also rebuild tokenizer.json to match

import shutil
from google.protobuf import text_format

# We need sentencepiece_model_pb2 — generate it from the .proto
!pip install -q sentencepiece protobuf

# Use sentencepiece's built-in protobuf
import sentencepiece.sentencepiece_model_pb2 as sp_pb2

# --- Load original SentencePiece model ---
original_sp_path = f"{LOCAL_MODEL_DIR}/tokenizer.model"

print(f"Loading original SentencePiece model: {original_sp_path}")
sp_model = sp_pb2.ModelProto()
with open(original_sp_path, "rb") as f:
    sp_model.ParseFromString(f.read())

orig_piece_count = len(sp_model.pieces)
print(f"  Original pieces: {orig_piece_count:,}")

# --- Build mapping from piece text to original index ---
# SentencePiece pieces are indexed 0..N-1, matching the token IDs
# We need to keep pieces at indices in keep_ids_sorted

# Create new model with only kept pieces
new_sp_model = sp_pb2.ModelProto()
new_sp_model.CopyFrom(sp_model)
del new_sp_model.pieces[:]  # Clear all pieces

kept_count = 0
skipped_count = 0

for new_id, old_id in enumerate(keep_ids_sorted):
    if old_id < orig_piece_count:
        piece = sp_pb2.ModelProto.SentencePiece()
        piece.CopyFrom(sp_model.pieces[old_id])
        new_sp_model.pieces.append(piece)
        kept_count += 1
    else:
        # This is an added token (beyond SentencePiece vocab) — handle below
        # Add a placeholder CONTROL piece
        piece = sp_pb2.ModelProto.SentencePiece()
        # Get the text from tokenizer
        token_text = tokenizer.convert_ids_to_tokens(old_id)
        piece.piece = token_text if token_text else f"<extra_id_{old_id}>"
        piece.score = 0.0
        piece.type = sp_pb2.ModelProto.SentencePiece.CONTROL
        new_sp_model.pieces.append(piece)
        kept_count += 1

print(f"  New pieces: {kept_count:,}")
print(f"  Removed: {orig_piece_count - kept_count + skipped_count:,}")

# Save new SentencePiece model
new_sp_path = f"{TRIMMED_MODEL_DIR}/tokenizer.model"
with open(new_sp_path, "wb") as f:
    f.write(new_sp_model.SerializeToString())
print(f"  ✅ Saved: {new_sp_path}")

# --- Rebuild tokenizer.json (HuggingFace fast tokenizer) ---
# Load original tokenizer.json
orig_tokenizer_json_path = f"{LOCAL_MODEL_DIR}/tokenizer.json"
if os.path.exists(orig_tokenizer_json_path):
    with open(orig_tokenizer_json_path, "r") as f:
        tok_json = json.load(f)

    # Update the vocab in the model section
    if "model" in tok_json and "vocab" in tok_json["model"]:
        old_vocab = tok_json["model"]["vocab"]
        new_vocab = {}

        # Build reverse mapping: old piece text -> old_id
        old_text_to_id = {}
        for piece_text, piece_id in old_vocab.items():
            old_text_to_id[piece_text] = piece_id

        # Create new vocab with only kept tokens
        for new_id, old_id in enumerate(keep_ids_sorted):
            # Find the piece text for this old_id
            for piece_text, pid in old_vocab.items():
                if pid == old_id:
                    new_vocab[piece_text] = new_id
                    break

        tok_json["model"]["vocab"] = new_vocab
        print(f"  tokenizer.json vocab: {len(old_vocab):,} → {len(new_vocab):,}")

    # Update added_tokens — remap IDs
    if "added_tokens" in tok_json:
        new_added = []
        for at in tok_json["added_tokens"]:
            old_id = at.get("id")
            if old_id in old_to_new:
                at_copy = dict(at)
                at_copy["id"] = old_to_new[old_id]
                new_added.append(at_copy)
        tok_json["added_tokens"] = new_added
        print(f"  added_tokens: {len(tok_json['added_tokens'])}")

    # Save updated tokenizer.json
    new_tok_json_path = f"{TRIMMED_MODEL_DIR}/tokenizer.json"
    with open(new_tok_json_path, "w") as f:
        json.dump(tok_json, f, ensure_ascii=False)
    print(f"  ✅ Saved: {new_tok_json_path}")
else:
    print(f"  ⚠️ No tokenizer.json found — will rely on tokenizer.model only")

# --- Copy other tokenizer config files ---
for fname in ["tokenizer_config.json", "special_tokens_map.json"]:
    src = f"{LOCAL_MODEL_DIR}/{fname}"
    dst = f"{TRIMMED_MODEL_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  Copied: {fname}")

# --- Update tokenizer_config.json with new vocab size ---
tok_config_path = f"{TRIMMED_MODEL_DIR}/tokenizer_config.json"
if os.path.exists(tok_config_path):
    with open(tok_config_path, "r") as f:
        tok_config = json.load(f)
    # Update added_tokens_decoder with new IDs
    if "added_tokens_decoder" in tok_config:
        new_decoder = {}
        for old_id_str, token_info in tok_config["added_tokens_decoder"].items():
            old_id = int(old_id_str)
            if old_id in old_to_new:
                new_decoder[str(old_to_new[old_id])] = token_info
        tok_config["added_tokens_decoder"] = new_decoder
    with open(tok_config_path, "w") as f:
        json.dump(tok_config, f, ensure_ascii=False, indent=2)
    print(f"  ✅ Updated: tokenizer_config.json")

# --- Validate: load trimmed tokenizer and test round-trip ---
print(f"\n--- Tokenizer round-trip validation ---")
try:
    trimmed_tokenizer = AutoTokenizer.from_pretrained(TRIMMED_MODEL_DIR)
    print(f"  Trimmed tokenizer vocab: {len(trimmed_tokenizer):,}")

    test_sentences = [
        "வணக்கம், எப்படி இருக்கிறீர்கள்?",
        "திருக்குறள் பற்றி சொல்லுங்கள்",
        "ரேஷன் கார்டு எப்படி பெறுவது?",
        "நீரிழிவு நோய் தடுப்பது எப்படி?",
        "Hello, how are you?",
        "Tamil Nadu government schemes",
        "TNPSC exam preparation tips",
        "சைபர் மோசடி புகார் 1930",
        "முதியோர் ஓய்வூதியம் Rs.1000",
        "COVID-19 vaccine registration",
    ]

    all_pass = True
    for sent in test_sentences:
        encoded = trimmed_tokenizer.encode(sent)
        decoded = trimmed_tokenizer.decode(encoded, skip_special_tokens=True)
        # Allow minor whitespace differences
        match = decoded.strip() == sent.strip()
        status = "✅" if match else "⚠️"
        if not match:
            all_pass = False
        print(f"  {status} '{sent[:40]}...' → {len(encoded)} tokens → '{decoded[:40]}...'")

    if all_pass:
        print(f"\n  ✅ All 10 round-trip tests PASSED")
    else:
        print(f"\n  ⚠️ Some round-trips had differences (may be whitespace normalization)")
        print(f"     This is usually acceptable — check outputs above")

except Exception as e:
    print(f"  ❌ Tokenizer load failed: {e}")
    print(f"     May need to fix tokenizer files manually")

Loading original SentencePiece model: ./vazhi-v7_1/tokenizer.model
  Original pieces: 262,144
  New pieces: 21,067
  Removed: 241,077
  ✅ Saved: ./vazhi-v7.1-trimmed/tokenizer.model
  tokenizer.json vocab: 262,144 → 21,067
  added_tokens: 6415
  ✅ Saved: ./vazhi-v7.1-trimmed/tokenizer.json
  Copied: tokenizer_config.json
  Copied: special_tokens_map.json
  ✅ Updated: tokenizer_config.json

--- Tokenizer round-trip validation ---
  ❌ Tokenizer load failed: Token `nt` out of vocabulary at line 1 column 11106570
     May need to fix tokenizer files manually


In [32]:
  import json

  with open('./vazhi-v7.1-trimmed/tokenizer.json', 'r') as f:
      tok_json = json.load(f)

  valid_tokens = set(tok_json['model']['vocab'].keys())

  # Filter merges — both parts AND merged result must be in trimmed vocab
  original_merges = tok_json['model']['merges']
  filtered_merges = []
  for merge in original_merges:
      a, b = merge[0], merge[1]
      merged = a + b
      if a in valid_tokens and b in valid_tokens and merged in valid_tokens:
          filtered_merges.append(merge)

  print(f"Merges: {len(original_merges)} → {len(filtered_merges)}")
  tok_json['model']['merges'] = filtered_merges

  # Fix added_tokens — drop any with id >= vocab_size
  NEW_VOCAB = len(tok_json['model']['vocab'])
  before = len(tok_json['added_tokens'])
  tok_json['added_tokens'] = [t for t in tok_json['added_tokens'] if t['id'] < NEW_VOCAB]
  print(f"Added tokens: {before} → {len(tok_json['added_tokens'])}")

  with open('./vazhi-v7.1-trimmed/tokenizer.json', 'w') as f:
      json.dump(tok_json, f, ensure_ascii=False)

  print("✅ Saved filtered tokenizer.json")

  #Then retry the validation:

  from transformers import AutoTokenizer
  tokenizer = AutoTokenizer.from_pretrained('./vazhi-v7.1-trimmed/')
  print(f"✅ Loaded! Vocab size: {tokenizer.vocab_size}")

  # Round-trip test
  test = "வணக்கம், எப்படி இருக்கீர்கள்?"
  ids = tokenizer.encode(test)
  decoded = tokenizer.decode(ids)
  print(f"Original: {test}")
  print(f"Decoded:  {decoded}")
  print(f"Match: {test == decoded}")


Merges: 514906 → 18440
Added tokens: 6415 → 6414
✅ Saved filtered tokenizer.json
✅ Loaded! Vocab size: 21067
Original: வணக்கம், எப்படி இருக்கீர்கள்?
Decoded:  <bos>வணக்கம், எப்படி இருக்கீர்கள்?
Match: False


In [26]:
llama-cli -m <q4_k_m.gguf> -p "வணக்கம்" -n 32

SyntaxError: invalid syntax (ipython-input-545904720.py, line 1)

In [34]:
  import torch, gc

  # Delete any previous models still in GPU memory
  for name in list(globals()):
      if isinstance(globals().get(name), torch.nn.Module):
          print(f"Deleting {name}")
          del globals()[name]

  gc.collect()
  torch.cuda.empty_cache()
  print(f"GPU free: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")


GPU free: 0.0 GB


In [35]:
# Cell 14 — Quick Quality Check (HF model, before GGUF)
#
# Run 10 Tamil + 3 English prompts through the trimmed HF model to verify
# quality BEFORE GGUF conversion. If quality drops significantly, we may
# need recovery SFT (Cell 15).

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(f"Loading trimmed model for quality check...")
trimmed_model = AutoModelForCausalLM.from_pretrained(
    TRIMMED_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map={"": 0} if torch.cuda.is_available() else "cpu",
)
trimmed_tokenizer = AutoTokenizer.from_pretrained(TRIMMED_MODEL_DIR)
trimmed_model.eval()

# Clear suppress_tokens if present
if hasattr(trimmed_model, 'generation_config') and hasattr(trimmed_model.generation_config, 'suppress_tokens'):
    trimmed_model.generation_config.suppress_tokens = None

print(f"  Vocab: {len(trimmed_tokenizer):,}")
print(f"  Params: {trimmed_model.num_parameters():,}")
print(f"  GPU: {torch.cuda.memory_allocated(0)/1024**3:.1f} GB" if torch.cuda.is_available() else "  CPU mode")

def run_hf_inference(model, tokenizer, user_text, max_new=200):
    """Run inference on HF model with Gemma 3 prompt format."""
    prompt = (
        f"<start_of_turn>user\n{SYSTEM_CONTEXT}\n\n{user_text}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=0.9,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0][input_len:]
    resp = tokenizer.decode(new_tokens, skip_special_tokens=True)
    for tok in ["<end_of_turn>", "<start_of_turn>"]:
        resp = resp.split(tok)[0]
    return resp.strip()

# Run eval on all prompts
print(f"\n{'='*65}")
print(f"  TRIMMED HF MODEL QUALITY CHECK")
print(f"{'='*65}")

trimmed_results = []
for item in ALL_PROMPTS:
    resp = run_hf_inference(trimmed_model, trimmed_tokenizer, item['text'])
    t_char = tamil_char_pct(resp)
    t_word, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)

    trimmed_results.append({
        'prompt': item['text'],
        'category': item['cat'],
        'response': resp,
        'tamil_char_pct': t_char,
        'tamil_word_pct': t_word,
        'repeat_ratio': rep,
    })

    print(f"\n  [{item['cat']:>10}] Char:{t_char:.0f}% Word:{t_word:.0f}% Rep:{rep:.2f}")
    print(f"    Q: {item['text']}")
    print(f"    A: {resp[:300]}")

# Summary
tamil_res = [r for r in trimmed_results if r['category'] != 'english']
avg_char = np.mean([r['tamil_char_pct'] for r in tamil_res])
avg_word = np.mean([r['tamil_word_pct'] for r in tamil_res])
avg_rep = np.mean([r['repeat_ratio'] for r in tamil_res])
non_empty = sum(1 for r in trimmed_results if len(r['response'].strip()) >= 10)

print(f"\n{'─'*65}")
print(f"  TRIMMED MODEL SUMMARY (Tamil prompts only):")
print(f"    Avg Tamil char: {avg_char:.1f}%")
print(f"    Avg Tamil word: {avg_word:.1f}%")
print(f"    Avg repeat:     {avg_rep:.2f}")
print(f"    Non-empty:      {non_empty}/{len(trimmed_results)}")
print(f"\n  Reference (v7.1 original): ~95% char, ~96% word (from training eval)")
print(f"  Quality drop threshold: >10% word score drop → trigger recovery SFT")

quality_ok = avg_word >= 85  # 96% - 10% threshold = 86%, using 85% as margin
if quality_ok:
    print(f"\n  ✅ GO — Quality preserved! Skip recovery SFT, proceed to GGUF.")
else:
    print(f"\n  ⚠️ Quality degraded — consider running recovery SFT (Cell 15)")

# Free GPU
del trimmed_model
gc.collect()
torch.cuda.empty_cache()

Loading trimmed model for quality check...


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 15.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 1.06 GiB is allocated by PyTorch, and 78.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Cell 15 — Optional: Recovery SFT (only run if Cell 14 shows quality degradation)
#
# If vocab trimming degraded quality >10%, run 1 epoch of LoRA SFT
# using the same config as v7.1 to recover Tamil capabilities.
#
# Config: LoRA r=16, q_proj+v_proj, LR 1e-5, 1 epoch, ~234 steps
#
# ⚠️  SKIP THIS CELL if Cell 14 shows quality_ok = True

SKIP_RECOVERY = quality_ok  # Set to False to force recovery SFT

if SKIP_RECOVERY:
    print("✅ Skipping recovery SFT — quality is preserved after trimming")
    print("   (Set SKIP_RECOVERY = False to force recovery SFT)")
else:
    print("Running recovery SFT on trimmed model...")
    print("Config matches v7.1: LoRA r=16, q_proj+v_proj, LR 1e-5, 1 epoch")

    !pip install -q trl peft

    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset

    # Load trimmed model on GPU
    recovery_model = AutoModelForCausalLM.from_pretrained(
        TRIMMED_MODEL_DIR,
        torch_dtype=torch.float16,
        device_map={"": 0},
    )
    recovery_tokenizer = AutoTokenizer.from_pretrained(TRIMMED_MODEL_DIR)

    # Apply LoRA (same config as v7.1)
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    recovery_model = get_peft_model(recovery_model, lora_config)
    recovery_model.print_trainable_parameters()

    # Load SFT v7.0 dataset
    sft_path = hf_hub_download(
        repo_id="CryptoYogi/vazhi-tamil-sft-v7_0",
        filename="vazhi-tamil-sft-v7_0-train.json",
        repo_type="dataset",
    )
    sft_train = json.load(open(sft_path))

    # Format for SFT: Gemma 3 chat format
    def format_for_sft(item):
        text = (
            f"<start_of_turn>user\n{SYSTEM_CONTEXT}\n\n"
            f"{item['instruction']}<end_of_turn>\n"
            f"<start_of_turn>model\n{item['output']}<end_of_turn>"
        )
        return {"text": text}

    train_ds = Dataset.from_list([format_for_sft(item) for item in sft_train])
    print(f"  Train samples: {len(train_ds)}")

    # Training config
    RECOVERY_DIR = "./vazhi-v7.1-trimmed-recovery"
    batch_size = 4 if torch.cuda.get_device_properties(0).total_memory > 20e9 else 2

    training_args = SFTConfig(
        output_dir=RECOVERY_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        max_seq_length=2048,
        dataset_text_field="text",
        logging_steps=10,
        save_strategy="epoch",
        fp16=True,
        optim="adamw_torch",
        seed=42,
    )

    trainer = SFTTrainer(
        model=recovery_model,
        args=training_args,
        train_dataset=train_ds,
    )

    print(f"\n🚀 Starting recovery SFT...")
    trainer.train()

    # Merge LoRA and save
    print(f"\nMerging LoRA adapter...")
    merged = recovery_model.merge_and_unload()
    merged.save_pretrained(TRIMMED_MODEL_DIR, safe_serialization=True)
    print(f"✅ Recovery SFT complete — merged model saved to {TRIMMED_MODEL_DIR}")

    del recovery_model, merged, trainer
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\n⚠️  Re-run Cell 14 to verify quality after recovery SFT")

In [39]:
  # Cell: Fix tokenizer.model for trimmed vocab
  # The SPM protobuf must match the 21K kept tokens

  import json
  from google.protobuf import text_format
  from sentencepiece import sentencepiece_model_pb2 as sp_model

  # Load the keep_ids (should already be in memory, otherwise reload)
  # keep_ids_sorted = sorted([id for id in keep_ids if id < 262144])

  # Load original SPM model
  spm = sp_model.ModelProto()
  with open("vazhi-v7.1-trimmed/tokenizer.model", "rb") as f:
      spm.ParseFromString(f.read())

  print(f"Original SPM pieces: {len(spm.pieces)}")

  # Build mapping: original_id -> keep (True/False)
  keep_set = set(keep_ids_sorted)

  # Filter SPM pieces: keep only those whose index is in keep_set
  # SPM pieces are indexed by position (piece[0] = token_id 0, piece[1] = token_id 1, etc.)
  new_pieces = []
  for idx, piece in enumerate(spm.pieces):
      if idx in keep_set:
          new_pieces.append(piece)

  # Clear and re-add filtered pieces
  del spm.pieces[:]
  spm.pieces.extend(new_pieces)

  print(f"Trimmed SPM pieces: {len(spm.pieces)}")

  # Save
  with open("vazhi-v7.1-trimmed/tokenizer.model", "wb") as f:
      f.write(spm.SerializeToString())

  print("✅ tokenizer.model rebuilt with trimmed vocab")

  # Also update tokenizer_config.json vocab_size if present
  import os
  tc_path = "vazhi-v7.1-trimmed/tokenizer_config.json"
  if os.path.exists(tc_path):
      with open(tc_path) as f:
          tc = json.load(f)
      if "vocab_size" in tc:
          tc["vocab_size"] = len(new_pieces)
          with open(tc_path, "w") as f:
              json.dump(tc, f, indent=2)
          print(f"Updated tokenizer_config.json vocab_size to {len(new_pieces)}")

  #After running this, re-run the GGUF conversion:

  # Re-convert to GGUF with fixed tokenizer
  !python llama.cpp/convert_hf_to_gguf.py vazhi-v7.1-trimmed/ \
      --outfile vazhi-v7.1-trimmed-f16.gguf \
      --outtype f16

  # Re-quantize
  !llama.cpp/build/bin/llama-quantize vazhi-v7.1-trimmed-f16.gguf \
      vazhi-v7.1-trimmed-q2_k.gguf Q2_K

  !llama.cpp/build/bin/llama-quantize vazhi-v7.1-trimmed-f16.gguf \
      vazhi-v7.1-trimmed-q4_k_m.gguf Q4_K_M


Original SPM pieces: 21067
Trimmed SPM pieces: 6228
✅ tokenizer.model rebuilt with trimmed vocab
INFO:hf-to-gguf:Loading model: vazhi-v7.1-trimmed
INFO:hf-to-gguf:Model architecture: Gemma3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.float16 --> F16, shape = {1152, 21067}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.float16 --> F32, shape = {1152}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.float16 --> F16, shape = {6912, 1152}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.float16 --> F16, shape = {1152, 6912}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.float16 --> F16, shape = {1152, 6912}
INFO:hf-to-gguf:blk.0.post_attention_norm.weight,  torch.float16 --> F32, shape = {1152}
INFO:hf-to-gguf:blk.0.post_ffw_norm.weight,        torch.float16 --> F32, 

In [40]:
# Cell 16 — GGUF Conversion + Quantization of Trimmed Model
#
# Convert the vocab-trimmed HF model to GGUF, then quantize to Q4_K_M.
# Expected: Q4_K_M ≈ 300 MiB (down from 762 MiB).

TRIMMED_F16_GGUF = "vazhi-v7.1-trimmed-f16.gguf"
TRIMMED_Q4_GGUF = "vazhi-v7.1-trimmed-q4_k_m.gguf"
TRIMMED_Q3_GGUF = "vazhi-v7.1-trimmed-q3_k_m.gguf"
TRIMMED_Q2_GGUF = "vazhi-v7.1-trimmed-q2_k.gguf"

# --- Convert trimmed model to f16 GGUF ---
print("Converting trimmed model to f16 GGUF...")
if not os.path.exists(TRIMMED_F16_GGUF):
    !python llama.cpp/convert_hf_to_gguf.py {TRIMMED_MODEL_DIR} \
        --outfile {TRIMMED_F16_GGUF} --outtype f16
else:
    print(f"  Already exists: {TRIMMED_F16_GGUF}")

if os.path.exists(TRIMMED_F16_GGUF):
    f16_size = os.path.getsize(TRIMMED_F16_GGUF)
    print(f"  ✅ {TRIMMED_F16_GGUF}: {f16_size/1e6:.1f} MB")
else:
    raise RuntimeError("f16 GGUF conversion failed — check logs above")

# --- Quantize to Q4_K_M, Q3_K_M, Q2_K ---
print(f"\nQuantizing trimmed model...")

trimmed_gguf_files = {}

for qt, outfile in [("Q4_K_M", TRIMMED_Q4_GGUF), ("Q3_K_M", TRIMMED_Q3_GGUF), ("Q2_K", TRIMMED_Q2_GGUF)]:
    print(f"\n  {qt} → {outfile}...", end=" ")
    if os.path.exists(outfile):
        print(f"exists ({os.path.getsize(outfile)/1e6:.0f} MB)")
    elif quantize(TRIMMED_F16_GGUF, outfile, qt.lower()):
        print(f"✅ ({os.path.getsize(outfile)/1e6:.0f} MB)")
    else:
        print("❌")
        continue
    trimmed_gguf_files[qt] = outfile

# --- Size comparison ---
print(f"\n\n{'='*70}")
print(f"SIZE COMPARISON: Original vs Trimmed")
print(f"{'='*70}")
print(f"  {'Model':<35} {'Size (MB)':>10} {'Size (MiB)':>11} {'4GB OK?':>8}")
print(f"  {'─'*35} {'─'*10} {'─'*11} {'─'*8}")

# Original sizes (from plan context)
orig_sizes = {
    "Original f16":     os.path.getsize(F16_GGUF) if os.path.exists(F16_GGUF) else 0,
    "Original Q4_K_M":  os.path.getsize("vazhi-v7.1-q4_k_m.gguf") if os.path.exists("vazhi-v7.1-q4_k_m.gguf") else 762_000_000,
    "Original Q3_K_M":  os.path.getsize("vazhi-v7.1-q3_k_m.gguf") if os.path.exists("vazhi-v7.1-q3_k_m.gguf") else 693_000_000,
    "Original Q2_K":    os.path.getsize("vazhi-v7.1-q2_k.gguf") if os.path.exists("vazhi-v7.1-q2_k.gguf") else 652_000_000,
}
trimmed_sizes = {
    "Trimmed f16":      os.path.getsize(TRIMMED_F16_GGUF) if os.path.exists(TRIMMED_F16_GGUF) else 0,
}
for qt, path in trimmed_gguf_files.items():
    trimmed_sizes[f"Trimmed {qt}"] = os.path.getsize(path)

for label, size_bytes in {**orig_sizes, **trimmed_sizes}.items():
    mb = size_bytes / 1e6
    mib = size_bytes / 1048576
    fits_4gb = "✅" if mib < 400 else ("⚠️" if mib < 600 else "❌")
    print(f"  {label:<35} {mb:>10.1f} {mib:>10.1f} {fits_4gb:>8}")

# Savings
if "Original Q4_K_M" in orig_sizes and "Trimmed Q4_K_M" in trimmed_gguf_files:
    orig_q4 = orig_sizes["Original Q4_K_M"]
    trim_q4 = os.path.getsize(trimmed_gguf_files["Q4_K_M"])
    saving = orig_q4 - trim_q4
    print(f"\n  Q4_K_M savings: {saving/1e6:.0f} MB ({100*saving/orig_q4:.0f}% reduction)")

Converting trimmed model to f16 GGUF...
  Already exists: vazhi-v7.1-trimmed-f16.gguf
  ✅ vazhi-v7.1-trimmed-f16.gguf: 1445.2 MB

Quantizing trimmed model...

  Q4_K_M → vazhi-v7.1-trimmed-q4_k_m.gguf... exists (505 MB)

  Q3_K_M → vazhi-v7.1-trimmed-q3_k_m.gguf... exists (421 MB)

  Q2_K → vazhi-v7.1-trimmed-q2_k.gguf... exists (389 MB)


SIZE COMPARISON: Original vs Trimmed
  Model                                Size (MB)  Size (MiB)  4GB OK?
  ─────────────────────────────────── ────────── ─────────── ────────
  Original f16                            2006.6     1913.6        ❌
  Original Q4_K_M                          806.1      768.7        ❌
  Original Q3_K_M                          722.4      688.9        ❌
  Original Q2_K                            689.8      657.9        ❌
  Trimmed f16                             1445.2     1378.2        ❌
  Trimmed Q4_K_M                           505.0      481.6       ⚠️
  Trimmed Q3_K_M                           421.4      401.9       ⚠

In [1]:
# Cell 17 — GGUF Quality Validation (Trimmed Q4_K_M)
#
# Final quality gate: run the same 13-prompt eval via llama.cpp CLI
# on the trimmed Q4_K_M GGUF. Compare against original Q4_K_M.

trimmed_q4_eval = eval_gguf(TRIMMED_Q4_GGUF, "Trimmed Q4_K_M")

# Compare against original Q4_K_M if available
orig_q4_path = "vazhi-v7.1-q4_k_m.gguf"
if os.path.exists(orig_q4_path):
    orig_q4_key = "Q4_K_M (baseline)"
    if orig_q4_key in all_eval_results:
        orig_q4_eval = all_eval_results[orig_q4_key]
    else:
        orig_q4_eval = eval_gguf(orig_q4_path, "Original Q4_K_M")

    # Side-by-side
    print(f"\n\n{'='*80}")
    print(f"  FINAL COMPARISON: Original Q4_K_M vs Trimmed Q4_K_M")
    print(f"{'='*80}")

    print(f"\n  {'Metric':<20} {'Original':>12} {'Trimmed':>12} {'Delta':>12}")
    print(f"  {'─'*20} {'─'*12} {'─'*12} {'─'*12}")

    metrics = [
        ("Size (MB)", orig_q4_eval['size_mb'], trimmed_q4_eval['size_mb']),
        ("Tamil char %", orig_q4_eval['avg_char'], trimmed_q4_eval['avg_char']),
        ("Tamil word %", orig_q4_eval['avg_word'], trimmed_q4_eval['avg_word']),
        ("Repeat ratio", orig_q4_eval['avg_rep'], trimmed_q4_eval['avg_rep']),
    ]

    for name, orig_val, trim_val in metrics:
        delta = trim_val - orig_val
        print(f"  {name:<20} {orig_val:>12.1f} {trim_val:>12.1f} {delta:>+12.1f}")

    # Per-prompt comparison
    print(f"\n  PER-PROMPT COMPARISON:")
    print(f"  {'─'*75}")
    for orig_r, trim_r in zip(orig_q4_eval['results'], trimmed_q4_eval['results']):
        print(f"\n  Q: {orig_r['prompt']}")
        print(f"  Original (W:{orig_r['tamil_word_pct']:.0f}%): {orig_r['response'][:150]}")
        print(f"  Trimmed  (W:{trim_r['tamil_word_pct']:.0f}%): {trim_r['response'][:150]}")
else:
    print(f"\n  ⚠️ Original Q4_K_M not available for comparison")

NameError: name 'eval_gguf' is not defined

In [38]:
# Cell 18 — Upload to HuggingFace + SHA256 Checksums
#
# Upload the best trimmed GGUF variants and the imatrix file.

import hashlib
from huggingface_hub import HfApi, create_repo

def sha256_file(path):
    """Compute SHA256 of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(8192)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

# --- Compute checksums ---
print("Computing SHA256 checksums...")
checksums = {}
upload_files = {}

# Trimmed GGUF variants
for label, path in trimmed_gguf_files.items():
    if os.path.exists(path):
        checksums[path] = sha256_file(path)
        upload_files[path] = path
        print(f"  {path}: {checksums[path][:16]}...")

# imatrix file
if os.path.exists(IMATRIX_FILE):
    checksums[IMATRIX_FILE] = sha256_file(IMATRIX_FILE)
    upload_files[IMATRIX_FILE] = IMATRIX_FILE

# Best imatrix GGUF (Q4_K_M imatrix for 6GB+ devices)
imat_q4 = "vazhi-v7.1-q4_k_m-imat.gguf"
if os.path.exists(imat_q4):
    checksums[imat_q4] = sha256_file(imat_q4)
    upload_files[imat_q4] = imat_q4

# --- Upload to HuggingFace ---
TRIMMED_REPO = "CryptoYogi/vazhi-v7_1-trimmed"
IMATRIX_REPO = "CryptoYogi/vazhi-v7_1-imatrix-GGUF"

print(f"\n--- Uploading trimmed model ---")
try:
    create_repo(TRIMMED_REPO, repo_type="model", exist_ok=True)
    api = HfApi()

    for local_path, remote_name in upload_files.items():
        if "trimmed" in local_path:
            print(f"  Uploading {local_path}...")
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=os.path.basename(local_path),
                repo_id=TRIMMED_REPO,
                repo_type="model",
            )
    # Also upload the trimmed HF model
    api.upload_folder(
        folder_path=TRIMMED_MODEL_DIR,
        repo_id=TRIMMED_REPO,
        repo_type="model",
        allow_patterns=["*.safetensors", "*.json", "*.model", "config.*"],
    )
    print(f"  ✅ Trimmed model uploaded: https://huggingface.co/{TRIMMED_REPO}")
except Exception as e:
    print(f"  ⚠️ Upload failed: {e}")
    print(f"     Upload manually: huggingface-cli upload {TRIMMED_REPO} {TRIMMED_MODEL_DIR}")

print(f"\n--- Uploading imatrix GGUFs ---")
try:
    create_repo(IMATRIX_REPO, repo_type="model", exist_ok=True)
    for local_path in upload_files:
        if "imat" in local_path or local_path == IMATRIX_FILE:
            print(f"  Uploading {local_path}...")
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=os.path.basename(local_path),
                repo_id=IMATRIX_REPO,
                repo_type="model",
            )
    print(f"  ✅ imatrix GGUFs uploaded: https://huggingface.co/{IMATRIX_REPO}")
except Exception as e:
    print(f"  ⚠️ Upload failed: {e}")

# --- Print checksums ---
print(f"\n{'='*70}")
print(f"SHA256 CHECKSUMS")
print(f"{'='*70}")
for path, checksum in checksums.items():
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {path:<45} {size_mb:>8.1f} MB")
    print(f"    {checksum}")

Computing SHA256 checksums...
  vazhi-v7.1-trimmed-q4_k_m.gguf: 7966ea73b9e3f9c6...
  vazhi-v7.1-trimmed-q3_k_m.gguf: 8c62b59fe7406914...
  vazhi-v7.1-trimmed-q2_k.gguf: 939d7c49f1249cf7...

--- Uploading trimmed model ---
  Uploading vazhi-v7.1-trimmed-q4_k_m.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-v7.1-trimmed-q4_k_m.gguf:   3%|3         | 15.9MB /  505MB            

  Uploading vazhi-v7.1-trimmed-q3_k_m.gguf...


...

  Uploading vazhi-v7.1-q4_k_m-imat.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  vazhi-v7.1-q4_k_m-imat.gguf :   4%|4         | 33.3MB /  806MB            

  ✅ imatrix GGUFs uploaded: https://huggingface.co/CryptoYogi/vazhi-v7_1-imatrix-GGUF

SHA256 CHECKSUMS
  vazhi-v7.1-trimmed-q4_k_m.gguf                   505.0 MB
    7966ea73b9e3f9c65ecc5eabc47bdcdfe3d049993409af47a3f6a7d00b88262b
  vazhi-v7.1-trimmed-q3_k_m.gguf                   421.4 MB
    8c62b59fe740691465c2526bde67c4fabe111831fe3bdfdba04122db8f054c7d
  vazhi-v7.1-trimmed-q2_k.gguf                     388.8 MB
    939d7c49f1249cf7911788142660abe0178a584a40c0301d69349ae00dd7eb90
  vazhi-imatrix.dat                                  1.5 MB
    7c1090e6d9a45727dd61bb58b7a668fcc8edcea680acd39149bb3b745b12ccc6
  vazhi-v7.1-q4_k_m-imat.gguf                      806.1 MB
    d8b7db1d1cda6a4ca491bfed3ee63115593833eb2f2a8e77cbce8748fe0aeb2c


In [2]:
# Cell 19 — Final Summary + Next Steps

print("=" * 75)
print("  VAZHI 4GB OPTIMIZATION — FINAL SUMMARY")
print("=" * 75)

# Test 0 Summary
print(f"\n  TEST 0: Gemma 3 270M-it Tamil Quality")
print(f"  {'─'*60}")
if results_270m:
    for label, r in sorted(results_270m.items(), key=lambda x: x[1]['size_mb']):
        print(f"    {label}: {r['avg_word']:.0f}% word, {r['avg_char']:.0f}% char, {r['size_mb']:.0f} MB")
    best = max(results_270m.values(), key=lambda x: x['avg_word'])
    verdict = "GO" if best['avg_word'] >= 70 else ("MARGINAL" if best['avg_word'] >= 40 else "NO-GO")
    print(f"    Verdict: {verdict} ({best['avg_word']:.0f}% Tamil word)")
else:
    print(f"    (Not run — execute Test 0 cell first)")

# Test 1 Summary (QAT)
print(f"\n  TEST 1: Gemma 3 1B QAT Tamil Quality")
print(f"  {'─'*60}")
if results_qat:
    for label, r in sorted(results_qat.items(), key=lambda x: x[1]['size_mb']):
        vs_v71 = r['size_mb'] - 806
        print(f"    {label}: {r['avg_word']:.0f}% word, {r['size_mb']:.0f} MB ({'+' if vs_v71>=0 else ''}{vs_v71:.0f} vs v7.1)")
    best_q = max(results_qat.values(), key=lambda x: x['avg_word'])
    print(f"    Best: {best_q['avg_word']:.0f}% Tamil word at {best_q['size_mb']:.0f} MB")
    # QAT Q2_K highlight
    q2k = results_qat.get("QAT Q2_K")
    if q2k:
        print(f"    QAT Q2_K: {q2k['avg_word']:.0f}% word at {q2k['size_mb']:.0f} MB (116MB < v7.1 Q4_K_M)")
else:
    print(f"    (Not run — execute Test 1 cell first)")

# Part A Summary
print(f"\n  PART A: imatrix + Embed/Output Quantization")
print(f"  {'─'*60}")
if all_eval_results:
    for qt in ["Q4_K_M", "Q3_K_M", "Q2_K"]:
        bl = all_eval_results.get(f"{qt} (baseline)")
        im = all_eval_results.get(f"{qt} (imatrix)")
        eq = all_eval_results.get(f"{qt} (embed_q8)")
        combo = all_eval_results.get(f"{qt} (imat+eq8)")
        if bl:
            line = f"    {qt}: baseline={bl['avg_word']:.0f}%w/{bl['size_mb']:.0f}MB"
            if im:
                line += f" | imat={im['avg_word']:.0f}%w/{im['size_mb']:.0f}MB"
            if eq:
                line += f" | eq8={eq['avg_word']:.0f}%w/{eq['size_mb']:.0f}MB"
            if combo:
                line += f" | both={combo['avg_word']:.0f}%w/{combo['size_mb']:.0f}MB"
            print(line)
else:
    print(f"    (Not run — execute Part A cells first)")

# Part B Summary
print(f"\n  PART B: Vocabulary Trimming")
print(f"  {'─'*60}")
print(f"    Original vocab:  {VOCAB_SIZE:,}")
print(f"    Trimmed vocab:   {NEW_VOCAB_SIZE:,} ({100*NEW_VOCAB_SIZE/VOCAB_SIZE:.1f}%)")

if trimmed_gguf_files:
    for qt, path in trimmed_gguf_files.items():
        size_mb = os.path.getsize(path) / 1e6
        print(f"    Trimmed {qt}: {size_mb:.0f} MB")

print(f"\n  QUALITY:")
if 'trimmed_q4_eval' in dir():
    print(f"    Trimmed Q4_K_M Tamil word: {trimmed_q4_eval['avg_word']:.1f}%")
    print(f"    Recovery SFT needed: {'No' if quality_ok else 'Yes'}")

# Decision matrix
print(f"\n  DECISION MATRIX")
print(f"  {'─'*60}")
print(f"  ┌───────────────────┬──────────────────────┬────────────────────────────────────────────────┐")
print(f"  │ 270M Tamil (T0)   │ QAT 1B Tamil (T1)    │ Action                                         │")
print(f"  ├───────────────────┼──────────────────────┼────────────────────────────────────────────────┤")
print(f"  │ Pass (≥70%)       │ QAT Q2K Tamil OK     │ 270M=4GB, QAT Q2K or Q4K=6GB+. Best case.     │")
print(f"  │ Pass              │ QAT Tamil weak       │ 270M=4GB, v7.1 imatrix=6GB+.                  │")
print(f"  │ Fail              │ QAT Q2K Tamil OK     │ Test QAT Q2K on 4GB device. QAT Q4K=6GB+.     │")
print(f"  │ Fail              │ QAT Q4K OK, Q2K bad  │ No 4GB LLM from QAT. QAT Q4K=6GB+.            │")
print(f"  │ Fail              │ QAT Tamil weak       │ v7.1 imatrix=6GB+. Vocab trim or hybrid=4GB.  │")
print(f"  │ Any               │ Any                  │ Native harness test to isolate Flutter overhead │")
print(f"  └───────────────────┴──────────────────────┴────────────────────────────────────────────────┘")

# Deployment recommendation
print(f"\n  DEPLOYMENT RECOMMENDATION")
print(f"  {'─'*60}")

# Determine best 4GB candidate
candidates_4gb = []
if results_270m:
    best270 = max(results_270m.values(), key=lambda x: x['avg_word'])
    if best270['avg_word'] >= 70:
        candidates_4gb.append(f"Gemma 3 270M Q4_K_M (~{best270['size_mb']:.0f} MB, {best270['avg_word']:.0f}% word)")
if results_qat:
    q2k = results_qat.get("QAT Q2_K")
    if q2k and q2k['avg_word'] >= 70:
        candidates_4gb.append(f"QAT Q2_K (~{q2k['size_mb']:.0f} MB, {q2k['avg_word']:.0f}% word)")

# Determine best 6GB+ candidate
candidates_6gb = []
if results_qat:
    qat_q4 = results_qat.get("QAT Q4_K_M")
    if qat_q4 and qat_q4['avg_word'] >= 85:
        candidates_6gb.append(f"QAT Q4_K_M (~{qat_q4['size_mb']:.0f} MB, {qat_q4['avg_word']:.0f}% word)")
if all_eval_results:
    best_imat = all_eval_results.get("Q4_K_M (imatrix)")
    if best_imat:
        candidates_6gb.append(f"v7.1 imatrix Q4_K_M (~{best_imat['size_mb']:.0f} MB, {best_imat['avg_word']:.0f}% word)")

print(f"    4GB devices (Samsung Galaxy A series, budget phones):")
if candidates_4gb:
    for i, c in enumerate(candidates_4gb):
        priority = "PRIMARY" if i == 0 else "BACKUP"
        print(f"      → {priority}: {c}")
else:
    print(f"      → No viable LLM candidate found")
if 'trimmed_q4_eval' in dir() and trimmed_gguf_files:
    trim_size = os.path.getsize(list(trimmed_gguf_files.values())[0]) / 1e6
    print(f"      → BACKUP: Vocab-trimmed Gemma 3 1B Q4_K_M (~{trim_size:.0f} MB)")
print(f"      → FALLBACK: Hybrid-only (SQLite deterministic lookups, no model)")
print(f"      → Gate LLM mode behind preflight memory/oom_score check")

print(f"")
print(f"    6GB+ devices (most mid-range phones 2023+):")
if candidates_6gb:
    for i, c in enumerate(candidates_6gb):
        priority = "PRIMARY" if i == 0 else "ALTERNATIVE"
        print(f"      → {priority}: {c}")
else:
    print(f"      → v7.1 Q4_K_M (~806 MB) — current deployment candidate")
    if all_eval_results and all_eval_results.get("Q4_K_M (imatrix)"):
        print(f"      → Upgrade to imatrix variant for better Tamil quality")

print(f"")
print(f"    All devices:")
print(f"      → Hybrid mode is always available (works offline, no model needed)")
print(f"      → 6 knowledge packs + voice I/O provide value without LLM")

# Flutter app integration
print(f"\n  FLUTTER APP INTEGRATION")
print(f"  {'─'*60}")
print(f"    1. Add new ModelVariant entries in lib/models/model_variant.dart:")
print(f"       ModelVariant.lite270m → Gemma 3 270M Q4_K_M (~250 MB)")
if candidates_4gb and 'QAT' in candidates_4gb[0]:
    print(f"       ModelVariant.qatLite → QAT Q2_K (~690 MB)")
print(f"       ModelVariant.qatFull → QAT Q4_K_M (~810 MB)")
print(f"    2. Add preflight memory check before offering LLM mode on 4GB devices")
print(f"    3. Update model_selector_sheet.dart with device-aware model suggestions")
print(f"    4. Physical test on 4GB Android device")
print(f"    5. If Flutter overhead is the blocker: android:process=\":inference\" for separate process")

# Next steps
print(f"\n  NEXT STEPS (priority order)")
print(f"  {'─'*60}")
print(f"    1. Human review of Test 0 + Test 1 outputs (metrics can lie)")
print(f"    2. Physical 4GB device test with best candidate GGUF")
print(f"    3. Native harness test (Termux + llama.cpp) to isolate Flutter overhead")
print(f"    4. If QAT Q2_K works on 4GB → do SFT on QAT base for VAZHI personality")
print(f"    5. If 270M works → light SFT for VAZHI personality")
print(f"    6. If both fail → ship vocab-trimmed 1B or hybrid-only for 4GB")
print(f"    7. Upload best GGUF candidates to HuggingFace")
print(f"    8. Update ModelRegistry with new variants + checksums")
print(f"    9. Update CLAUDE.md + TRAINING_LOG.md with results")
print(f"   10. Watch for IndiaAI Mission models (GenLoop Yukti/Varta/Kavach) — purpose-built for this use case")

print(f"\n{'='*75}")

  VAZHI 4GB OPTIMIZATION — FINAL SUMMARY

  TEST 0: Gemma 3 270M-it Tamil Quality
  ────────────────────────────────────────────────────────────


NameError: name 'results_270m' is not defined